Imports

In [17]:
!pip install spacy

In [18]:
!pip install --upgrade typing_extensions

In [19]:
# Download directly using pip
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 3.4 MB/s eta 0:00:04
     ---- ----------------------------------- 1.3/12.8 MB 3.0 MB/s eta 0:00:04
     ------ --------------------------------- 2.1/12.8 MB 3.0 MB/s eta 0:00:04
     ------- -------------------------------- 2.4/12.8 MB 2.6 MB/s eta 0:00:04
     --------- ------------------------------ 2.9/12.8 MB 2.5 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 2.3 MB/s eta 0:00:05
     --------- ------------------------------ 3.1/12.8 MB 2.3 MB/s eta 0:00:05
     ---------- ----------------------------- 3.4/12.8 MB 2.0 MB/s eta 0:00:05
     ---------- ----------------------------- 3.4/12.8 MB 2.0 MB/s eta 0:00:05
     ---------- ----------------------------- 3.4/12.8 MB 2.0 MB/s eta 0:00:05
     ----------- ---------------------------- 3.7/12.8 MB 1.5 MB/s

In [20]:
import sys
!{sys.executable} -m pip install groq neo4j

In [21]:
import sys
!{sys.executable} -m pip install sentence_transformers

In [22]:
import sys
!{sys.executable} -m pip install faiss-cpu

In [23]:
import sys
!{sys.executable} -m pip install --upgrade pip

In [24]:
import re
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import spacy
nlp = spacy.load('en_core_web_sm')
from spacy.matcher import PhraseMatcher
from enum import Enum
from groq import Groq
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
import numpy as np
from tqdm import tqdm
import faiss

1. INPUT PREPROCESSING

In [25]:
class Intent(Enum):
    """Hotel travel assistant intent types"""
    HOTEL_SEARCH = "HOTEL_SEARCH"
    HOTEL_RECOMMENDATION = "HOTEL_RECOMMENDATION"
    REVIEW_QUERY = "REVIEW_QUERY"
    HOTEL_COMPARISON = "HOTEL_COMPARISON"
    VISA_INQUIRY = "VISA_INQUIRY"
    LOCATION_BASED_QUERY = "LOCATION_BASED_QUERY"
    TRAVELER_PREFERENCE_QUERY = "TRAVELER_PREFERENCE_QUERY"
    RATING_ANALYSIS = "RATING_ANALYSIS"
    STATISTICAL_QUERY = "STATISTICAL_QUERY"
    DEMOGRAPHIC_RECOMMENDATION = "DEMOGRAPHIC_RECOMMENDATION"
    UNKNOWN = "UNKNOWN"

ENTITY EXTRACTION

In [26]:
class EntityExtractor:
    """
    Extract entities from user queries using hybrid approach:
    - spaCy PhraseMatcher for named entities
    - Regex for structured patterns
    """

    def __init__(self, neo4j_driver=None):
        """
        Initialize entity extractor.

        Args:
            neo4j_driver: Optional Neo4j driver to fetch entities from database
        """
        # Load spaCy model
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:
            print("Downloading spaCy model...")
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
            self.nlp = spacy.load("en_core_web_sm")

        # Initialize entity lists
        if neo4j_driver:
            # Fetch from Neo4j (production)
            self.cities = self._fetch_cities_from_neo4j(neo4j_driver)
            self.countries = self._fetch_countries_from_neo4j(neo4j_driver)
            self.hotels = self._fetch_hotels_from_neo4j(neo4j_driver)
        else:
            # Use sample lists (development/testing)
            self.cities = [
              'new york', 'nyc', 'london', 'paris', 'tokyo', 'dubai',
              'singapore', 'sydney', 'rio de janeiro', 'rio', 'berlin',
              'toronto', 'shanghai', 'mexico city', 'mumbai', 'rome',
              'cape town', 'seoul', 'moscow', 'cairo', 'barcelona',
              'bangkok', 'istanbul', 'amsterdam', 'buenos aires', 'lagos',
              'wellington'
            ]
            self.countries = [
              'united states', 'usa', 'us','united kingdom', 'uk', 'france', 'japan',
              'united arab emirates', 'uae', 'singapore', 'australia', 'brazil',
              'germany', 'canada', 'china', 'mexico', 'india', 'italy',
              'south africa', 'south korea', 'russia', 'egypt', 'spain',
              'thailand', 'turkey', 'netherlands', 'argentina', 'nigeria',
              'new zealand'
            ]
            self.hotels = [
              'the azure tower', 'the royal compass', "l'étoile palace",
              'kyo-to grand', 'the golden oasis', 'marina bay zenith',
              'sydney harbour grand', 'copacabana lux', 'berlin mitte elite',
              'the maple grove', 'the bund palace', 'aztec heights',
              'the gateway royale', 'colosseum gardens', 'table mountain view',
              'han river oasis', 'kremlin suites', 'nile grandeur',
              "gaudi's retreat", 'the orchid palace', 'the bosphorus inn',
              'canal house grand', 'tango boutique', 'the savannah house',
              'the kiwi grand'
            ]

        # Traveler types (domain knowledge)
        self.traveler_types = [
            'business traveler', 'business', 'solo traveler', 'solo',
            'family', 'couple'
        ]

        # Setup spaCy matchers
        self._setup_matchers()

    def _fetch_cities_from_neo4j(self, driver):
        """Fetch cities from Neo4j database"""
        query = "MATCH (c:City) RETURN DISTINCT c.name as name"
        with driver.session() as session:
            result = session.run(query)
            return [record["name"].lower() for record in result]

    def _fetch_countries_from_neo4j(self, driver):
        """Fetch countries from Neo4j database"""
        query = "MATCH (c:Country) RETURN DISTINCT c.name as name"
        with driver.session() as session:
            result = session.run(query)
            countries = [record["name"].lower() for record in result]

        # Add common abbreviations
        abbreviations = {
            'usa': 'united states',
            'us': 'united states',
            'uk': 'united kingdom',
            'uae': 'united arab emirates'
        }
        countries.extend(abbreviations.keys())
        return countries

    def _fetch_hotels_from_neo4j(self, driver):
        """Fetch hotels from Neo4j database"""
        query = "MATCH (h:Hotel) RETURN DISTINCT h.name as name"
        with driver.session() as session:
            result = session.run(query)
            return [record["name"].lower() for record in result]

    def _setup_matchers(self):
        """Setup PhraseMatcher for named entities"""
        # Create matchers
        self.city_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.country_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.hotel_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.traveler_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")

        # Add patterns (sort by length descending to match longest phrases first)
        city_patterns = [self.nlp.make_doc(city) for city in sorted(self.cities, key=len, reverse=True)]
        country_patterns = [self.nlp.make_doc(country) for country in sorted(self.countries, key=len, reverse=True)]
        hotel_patterns = [self.nlp.make_doc(hotel) for hotel in sorted(self.hotels, key=len, reverse=True)]
        traveler_patterns = [self.nlp.make_doc(t) for t in sorted(self.traveler_types, key=len, reverse=True)]

        self.city_matcher.add("CITY", city_patterns)
        self.country_matcher.add("COUNTRY", country_patterns)
        self.hotel_matcher.add("HOTEL", hotel_patterns)
        self.traveler_matcher.add("TRAVELER_TYPE", traveler_patterns)

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """
        Extract all entities from text using hybrid approach.

        Args:
            text: User query text

        Returns:
            Dictionary of entity types and their values
        """
        entities = defaultdict(list)

        # Preprocess text
        text_lower = text.lower().strip()

        # Process with spaCy
        doc = self.nlp(text_lower)

        # === SPACY PHRASEMATCHER: For named entities ===

        # Extract cities
        city_matches = self.city_matcher(doc)
        for match_id, start, end in city_matches:
            entities['city'].append(doc[start:end].text)

        # Extract countries
        country_matches = self.country_matcher(doc)
        for match_id, start, end in country_matches:
            entities['country'].append(doc[start:end].text)

        # Extract hotels
        hotel_matches = self.hotel_matcher(doc)
        for match_id, start, end in hotel_matches:
            entities['hotel'].append(doc[start:end].text)

        # Extract traveler types
        traveler_matches = self.traveler_matcher(doc)
        for match_id, start, end in traveler_matches:
            entities['traveler_type'].append(doc[start:end].text)

        # === REGEX: For structured patterns ===

        # Age patterns
        age_pattern = r'\b(\d+)\s*(?:year|yr)s?\s*old\b'
        age_matches = re.findall(age_pattern, text_lower)
        if age_matches:
            entities['age'].extend(age_matches)

        # Star rating patterns
        star_pattern = r'\b(\d)\s*[-\s]?star\b'
        star_matches = re.findall(star_pattern, text_lower)
        if star_matches:
            entities['star_rating'].extend(star_matches)

        # Gender patterns
        if re.search(r'\b(?:male|man|men|boy)\b', text_lower):
            entities['gender'].append('male')
        if re.search(r'\b(?:female|woman|women|girl|lady)\b', text_lower):
            entities['gender'].append('female')

        # Hotel features (using spaCy lemmatization)
        feature_keywords = {
            'clean': 'cleanliness',
            'cleanliness': 'cleanliness',
            'comfort': 'comfort',
            'comfortable': 'comfort',
            'facility': 'facilities',
            'facilities': 'facilities',
            'amenity': 'facilities',
            'amenities': 'facilities',
            'location': 'location',
            'staff': 'staff',
            'service': 'staff',
            'value': 'value_for_money',
            'price': 'value_for_money',
            'money': 'value_for_money'
        }

        for token in doc:
            if token.lemma_ in feature_keywords:
                entities['feature'].append(feature_keywords[token.lemma_])

        # Remove duplicates while preserving order
        for key in entities:
            seen = set()
            entities[key] = [x for x in entities[key] if not (x in seen or seen.add(x))]

        return dict(entities)

INTENT CLASSIFICATION

LLM Based

In [27]:
client = Groq(api_key="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")


# --- Prompt ---
HOTEL_INTENT_PROMPT = """
You are an intent classifier for a Hotel Travel Assistant.
Your task: read the user query and return ONLY ONE intent label from this list:

HOTEL_SEARCH
HOTEL_RECOMMENDATION
REVIEW_QUERY
HOTEL_COMPARISON
VISA_INQUIRY
LOCATION_BASED_QUERY
TRAVELER_PREFERENCE_QUERY
RATING_ANALYSIS
STATISTICAL_QUERY
DEMOGRAPHIC_RECOMMENDATION

Rules:
- Return ONLY the label.
- No sentences.
- No explanation.
- No JSON.

User Query: "{query}"
"""


# --- Classifier function ---
def classify_hotel_intent(query: str) -> str:
    prompt = HOTEL_INTENT_PROMPT.format(query=query)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10
    )

    return response.choices[0].message.content.strip()

In [28]:
# --- TESTS ---
print(classify_hotel_intent("Find hotels in Paris"))
print(classify_hotel_intent("Recommend a hotel for a 28-year-old female solo traveler"))
print(classify_hotel_intent("Do I need a visa to travel from Egypt to France?"))
print(classify_hotel_intent("What do people say about the Hilton in Cairo?"))
print(classify_hotel_intent("Compare hotels in Rome vs Florence"))
print(classify_hotel_intent("What's the average rating of 5-star hotels?"))
print(classify_hotel_intent("Show me hotels with good cleanliness ratings in Dubai"))
print(classify_hotel_intent("Which hotels do young male travelers prefer?"))
print(classify_hotel_intent("Tell me about the Marriott Downtown"))
print(classify_hotel_intent("Hello, can you help me?"))
print(classify_hotel_intent("What cities have the most hotels?"))
print(classify_hotel_intent("I'm a 45 year old business traveler, suggest hotels in London"))
print(classify_hotel_intent("Find 5-star hotels in New York for families"))
print(classify_hotel_intent("Hotels with best staff ratings in Los Angeles"))
print(classify_hotel_intent("Compare Hilton vs Marriott in Paris"))

HOTEL_SEARCH
HOTEL_RECOMMENDATION
VISA_INQUIRY
REVIEW_QUERY
HOTEL_COMPARISON
RATING_ANALYSIS
HOTEL_SEARCH
TRAVELER_PREFERENCE_QUERY
HOTEL_SEARCH
HOTEL_SEARCH
STATISTICAL_QUERY
HOTEL_RECOMMENDATION
HOTEL_SEARCH
HOTEL_COMPARISON
HOTEL_COMPARISON


Rule Based

In [29]:
class HotelIntentClassifier:
    """
    Classify user intent for hotel travel queries using rule-based approach.
    Integrated with entity extraction.
    """

    def __init__(self, neo4j_driver=None):
        """
        Initialize intent classifier.

        Args:
            neo4j_driver: Optional Neo4j driver for entity extraction
        """
        self.entity_extractor = EntityExtractor(neo4j_driver)

        # Define intent patterns with keywords and weights
        self.intent_patterns = {
            Intent.HOTEL_SEARCH: {
                'keywords': ['find', 'search', 'show', 'list', 'hotels in', 'hotels near', 'available'],
            },
            Intent.HOTEL_RECOMMENDATION: {
                'keywords': ['recommend', 'suggest', 'best hotel', 'good hotel', 'looking for'],
            },
            Intent.REVIEW_QUERY: {
                'keywords': ['review', 'rating', 'what do people say', 'opinion', 'feedback', 'comment'],
            },
            Intent.HOTEL_COMPARISON: {
                'keywords': ['compare', 'versus', 'vs', 'difference', 'between', 'better'],
            },
            Intent.VISA_INQUIRY: {
                'keywords': ['visa', 'visa requirement', 'need visa', 'travel from', 'passport'],
            },
            Intent.LOCATION_BASED_QUERY: {
                'keywords': ['cities', 'countries', 'destinations', 'where', 'location'],
            },
            Intent.TRAVELER_PREFERENCE_QUERY: {
                'keywords': ['travelers prefer', 'popular with', 'suited for', 'type of traveler'],
            },
            Intent.RATING_ANALYSIS: {
                'keywords': ['best rated', 'highest rating', 'top rated', 'rating for'],
            },
            Intent.STATISTICAL_QUERY: {
                'keywords': ['average', 'how many', 'total', 'count', 'statistics', 'distribution'],
            },
            Intent.DEMOGRAPHIC_RECOMMENDATION: {
                'keywords': ['year old', 'years old', 'age', 'male', 'female', 'for me'],
            },
        }

    def preprocess_text(self, text: str) -> str:
        """
        Preprocess text for intent classification.

        Args:
            text: Raw user query

        Returns:
            Preprocessed text
        """
        # Convert to lowercase and strip whitespace
        text = text.lower().strip()

        # Remove extra whitespace
        text = ' '.join(text.split())

        return text

    def classify(self, text: str) -> Tuple[Intent, float, Dict]:
        """
        Classify user intent and extract entities.

        Args:
            text: User query text

        Returns:
            Tuple of (intent, confidence, metadata)
            - intent: Intent enum value
            - confidence: Confidence score (0-1)
            - metadata: Dictionary containing entities and other info
        """
        # Preprocess text
        preprocessed = self.preprocess_text(text)

        # Extract entities
        entities = self.entity_extractor.extract_entities(text)

        # Calculate intent scores
        intent_scores = {}

        for intent, pattern in self.intent_patterns.items():
            score = 0
            keyword_matches = 0

            for keyword in pattern['keywords']:
                if keyword in preprocessed:
                    keyword_matches += 1

            if keyword_matches > 0:
                # Base score from keyword matches
                score = min(keyword_matches * 0.2, 0.7)

                # Boost score based on entities (entity presence indicates relevance)
                entity_boost = self._calculate_entity_boost(intent, entities)
                score = min(1.0, score + entity_boost)

                intent_scores[intent] = score

        # Special case: demographic recommendation if age/gender found
        if ('age' in entities or 'gender' in entities) and 'traveler_type' in entities:
            if Intent.DEMOGRAPHIC_RECOMMENDATION not in intent_scores or \
               intent_scores.get(Intent.DEMOGRAPHIC_RECOMMENDATION, 0) < 0.8:
                intent_scores[Intent.DEMOGRAPHIC_RECOMMENDATION] = 0.80

        # Special case: visa inquiry if multiple countries mentioned
        if len(entities.get('country', [])) >= 2:
            if Intent.VISA_INQUIRY not in intent_scores or \
               intent_scores.get(Intent.VISA_INQUIRY, 0) < 0.7:
                intent_scores[Intent.VISA_INQUIRY] = 0.70

        # Get best intent
        if intent_scores:
            best_intent = max(intent_scores.items(), key=lambda x: x[1])
            intent = best_intent[0]
            confidence = best_intent[1]
        else:
            intent = Intent.UNKNOWN
            confidence = 0.0

        # Prepare metadata
        metadata = {
            'entities': entities,
            'preprocessed_text': preprocessed,
            'original_text': text,
            'all_intent_scores': dict(intent_scores)
        }

        return intent, confidence, metadata

    def _calculate_entity_boost(self, intent: Intent, entities: Dict) -> float:
        """
        Calculate confidence boost based on entity presence for specific intents.

        Args:
            intent: Current intent being evaluated
            entities: Extracted entities

        Returns:
            Boost value (0-0.3)
        """
        boost = 0.0

        # Different intents benefit from different entities
        if intent == Intent.HOTEL_SEARCH:
            if 'city' in entities or 'country' in entities:
                boost += 0.15
            if 'star_rating' in entities or 'feature' in entities:
                boost += 0.10

        elif intent == Intent.HOTEL_RECOMMENDATION:
            if 'traveler_type' in entities:
                boost += 0.15
            if 'city' in entities:
                boost += 0.10

        elif intent == Intent.REVIEW_QUERY:
            if 'hotel' in entities or 'city' in entities:
                boost += 0.15

        elif intent == Intent.VISA_INQUIRY:
            if 'country' in entities and len(entities['country']) >= 1:
                boost += 0.20

        elif intent == Intent.DEMOGRAPHIC_RECOMMENDATION:
            if 'age' in entities or 'gender' in entities:
                boost += 0.15
            if 'traveler_type' in entities:
                boost += 0.10

        return min(0.3, boost)  # Cap at 0.3

Testing

In [30]:
# Test entity extraction independently
print("=" * 80)
print("ENTITY EXTRACTION TESTS")
print("=" * 80)

extractor = EntityExtractor()

test_queries_entities = [
    "Find 5-star hotels in New York",
    "I'm a 28 year old male business traveler",
    "Hotels with good cleanliness in Los Angeles",
    "Compare hotels in United States vs United Kingdom",
    "Do I need a visa to travel from Egypt to France?"
]

for query in test_queries_entities:
    entities = extractor.extract_entities(query)
    print(f"\nQuery: {query}")
    print(f"Entities: {entities}")
    print("-" * 80)

ENTITY EXTRACTION TESTS

Query: Find 5-star hotels in New York
Entities: {'city': ['new york'], 'star_rating': ['5']}
--------------------------------------------------------------------------------

Query: I'm a 28 year old male business traveler
Entities: {'traveler_type': ['business', 'business traveler'], 'age': ['28'], 'gender': ['male']}
--------------------------------------------------------------------------------

Query: Hotels with good cleanliness in Los Angeles
Entities: {'feature': ['cleanliness']}
--------------------------------------------------------------------------------

Query: Compare hotels in United States vs United Kingdom
Entities: {'country': ['united states', 'united kingdom']}
--------------------------------------------------------------------------------

Query: Do I need a visa to travel from Egypt to France?
Entities: {'country': ['egypt', 'france']}
--------------------------------------------------------------------------------


In [31]:
# Test complete intent classification pipeline
print("\n" + "=" * 80)
print("INTENT CLASSIFICATION + ENTITY EXTRACTION - INTEGRATED TESTS")
print("=" * 80)

classifier = HotelIntentClassifier()

test_queries = [
    "Find hotels in Paris",
    "Recommend a hotel for a 28-year-old female solo traveler",
    "Do I need a visa to travel from Egypt to France?",
    "What do people say about the Hilton in Cairo?",
    "Compare hotels in Rome vs Florence",
    "What's the average rating of 5-star hotels?",
    "Show me hotels with good cleanliness ratings in Dubai",
    "Which hotels do young male travelers prefer?",
    "Tell me about the Marriott Downtown",
    "Hello, can you help me?",
    "What cities have the most hotels?",
    "I'm a 45 year old business traveler, suggest hotels in London",
    "Find 5-star hotels in New York for families",
    "Hotels with best staff ratings in Los Angeles",
    "Compare Hilton vs Marriott in Paris"
]

for query in test_queries:
    intent, confidence, metadata = classifier.classify(query)

    print(f"\nQuery: {query}")
    print(f"Intent: {intent.value}")
    print(f"Confidence: {confidence:.2f}")
    print(f"Entities: {metadata['entities']}")

    # Show top 3 intent scores for transparency
    all_scores = metadata['all_intent_scores']
    if all_scores:
        top_3 = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"Top 3 intents: {[(i.value, f'{s:.2f}') for i, s in top_3]}")

    print("-" * 80)


INTENT CLASSIFICATION + ENTITY EXTRACTION - INTEGRATED TESTS

Query: Find hotels in Paris
Intent: HOTEL_SEARCH
Confidence: 0.55
Entities: {'city': ['paris']}
Top 3 intents: [('HOTEL_SEARCH', '0.55')]
--------------------------------------------------------------------------------

Query: Recommend a hotel for a 28-year-old female solo traveler
Intent: DEMOGRAPHIC_RECOMMENDATION
Confidence: 0.80
Entities: {'traveler_type': ['solo', 'solo traveler'], 'gender': ['female']}
Top 3 intents: [('DEMOGRAPHIC_RECOMMENDATION', '0.80'), ('HOTEL_RECOMMENDATION', '0.35')]
--------------------------------------------------------------------------------

Query: Do I need a visa to travel from Egypt to France?
Intent: VISA_INQUIRY
Confidence: 0.70
Entities: {'country': ['egypt', 'france']}
Top 3 intents: [('VISA_INQUIRY', '0.70')]
--------------------------------------------------------------------------------

Query: What do people say about the Hilton in Cairo?
Intent: REVIEW_QUERY
Confidence: 0.35


USER INPUT EMBEDDING

In [32]:
# Model 1: SBERT (small, fast)
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Model 2: BGE (larger, better quality)
bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5')

In [33]:
def embed_query(query_text):
    """
    Embed the same query using both SBERT and BGE.

    Args:
        query_text: User query string

    Returns:
        Dictionary with embeddings from both models
    """

    # SBERT embedding (384 dims)
    sbert_embedding = sbert_model.encode(query_text)

    # BGE embedding (1024 dims)
    bge_embedding = bge_model.encode(query_text)

    return {
        'sbert': sbert_embedding,
        'bge': bge_embedding
    }

In [34]:
query = "Find 5-star hotels in Paris with excellent cleanliness"

embeddings = embed_query(query)

print(f"Query: {query}\n")

print(f"SBERT embedding shape: {embeddings['sbert'].shape}")
print(f"SBERT (first 5): {embeddings['sbert'][:5]}\n")

print(f"BGE embedding shape: {embeddings['bge'].shape}")
print(f"BGE (first 5): {embeddings['bge'][:5]}")

Query: Find 5-star hotels in Paris with excellent cleanliness

SBERT embedding shape: (384,)
SBERT (first 5): [ 0.07261946 -0.04525768  0.07977296  0.00909202  0.04659879]

BGE embedding shape: (1024,)
BGE (first 5): [-0.00637323  0.02636962  0.01333172  0.04589355 -0.01661473]


2. GRAPH RETRIEVAL LAYER

BASELINE

In [35]:
class QueryMapper:
    """Maps extracted entities to query parameters for different intents."""
    
    def map_entities_to_parameters(self, intent: str, entities: dict) -> dict:
        """
        Convert extracted entities into query parameters for the selected intent.
        
        Args:
            intent: The classified intent (e.g., 'HOTEL_SEARCH')
            entities: Dictionary of extracted entities
            
        Returns:
            Dictionary of parameters for the Cypher query
        """
        params = {}
        
        # Common mappings used across multiple intents
        if 'city' in entities and entities['city']:
            params['city'] = entities['city'][0]  # Take first match
        
        if 'country' in entities and entities['country']:
            params['country'] = entities['country'][0]
        
        if 'hotel' in entities and entities['hotel']:
            params['hotel_name'] = entities['hotel'][0]
        
        # Intent-specific mappings
        if intent == 'HOTEL_SEARCH':
            # Star rating
            if 'star_rating' in entities and entities['star_rating']:
                params['min_star_rating'] = int(entities['star_rating'][0])
            
            # Features (cleanliness, comfort, facilities)
            if 'feature' in entities and entities['feature']:
                params['feature_type'] = entities['feature'][0]
                params['min_feature_score'] = 4.0  # Threshold for "good" features
            
            params['limit'] = 10
        
        elif intent == 'HOTEL_RECOMMENDATION':
            if 'feature' in entities and entities['feature']:
                params['feature_preference'] = entities['feature'][0]
                params['min_score'] = 4.0
            
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            
            params['limit'] = 10
        
        elif intent == 'REVIEW_QUERY':
            params['limit'] = 20
            
            # Score filtering
            params['min_score'] = None
            params['max_score'] = None
            
            # Demographic filters
            if 'gender' in entities and entities['gender']:
                params['gender'] = entities['gender'][0].capitalize()
            
            if 'age' in entities and entities['age']:
                age = int(entities['age'][0])
                params['age_min'] = age - 5
                params['age_max'] = age + 5
            
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
        
        elif intent == 'HOTEL_COMPARISON':
            if 'hotel' in entities and len(entities['hotel']) >= 2:
                params['hotel_name_1'] = entities['hotel'][0]
                params['hotel_name_2'] = entities['hotel'][1]
                params['city_1'] = None
                params['city_2'] = None
            elif 'city' in entities and len(entities['city']) >= 2:
                params['hotel_name_1'] = None
                params['hotel_name_2'] = None
                params['city_1'] = entities['city'][0]
                params['city_2'] = entities['city'][1]
            elif 'hotel' in entities and len(entities['hotel']) == 1:
                params['hotel_name_1'] = entities['hotel'][0]
                params['hotel_name_2'] = None
                params['city_1'] = None
                params['city_2'] = None
        
        elif intent == 'VISA_INQUIRY':
            if 'country' in entities and len(entities['country']) >= 2:
                params['from_country'] = entities['country'][0]
                params['to_country'] = entities['country'][1]
            elif 'country' in entities and len(entities['country']) == 1:
                params['from_country'] = entities['country'][0]
                params['to_country'] = None
        
        elif intent == 'LOCATION_BASED_QUERY':
            if 'star_rating' in entities and entities['star_rating']:
                params['min_star_rating'] = int(entities['star_rating'][0])
            
            params['min_hotels'] = 1
            params['limit'] = 20
        
        elif intent == 'TRAVELER_PREFERENCE_QUERY':
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            
            if 'age' in entities and entities['age']:
                age = int(entities['age'][0])
                params['age_min'] = age - 5
                params['age_max'] = age + 5
            
            if 'gender' in entities and entities['gender']:
                params['gender'] = entities['gender'][0].capitalize()
            
            params['min_travelers'] = 2
            params['limit'] = 10
        
        elif intent == 'RATING_ANALYSIS':
            if 'star_rating' in entities and entities['star_rating']:
                params['star_rating'] = int(entities['star_rating'][0])
            
            params['min_reviews'] = 1
            params['sort_by'] = 'overall'
            params['limit'] = 10
        
        elif intent == 'STATISTICAL_QUERY':
            if 'star_rating' in entities and entities['star_rating']:
                params['star_rating'] = int(entities['star_rating'][0])
            
            params['limit'] = 20
        
        elif intent == 'DEMOGRAPHIC_RECOMMENDATION':
            if 'age' in entities and entities['age']:
                params['age'] = int(entities['age'][0])
            else:
                params['age'] = 30  # Default age
            
            params['age_range'] = 5  # +/- 5 years
            
            if 'gender' in entities and entities['gender']:
                params['gender'] = entities['gender'][0].capitalize()
            
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            
            params['min_similar_travelers'] = 2
            params['limit'] = 10
        
        return params


In [36]:
class QueryExecutor:
    """Executes Cypher queries against Neo4j database."""
    
    def __init__(self, uri: str, user: str, password: str):
        """
        Initialize the query executor with Neo4j connection.
        
        Args:
            uri: Neo4j connection URI (e.g., 'neo4j://localhost:7687')
            user: Database username
            password: Database password
        """
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.query_templates = self._load_query_templates()
    
    def _load_query_templates(self) -> Dict[str, str]:
        """Load all Cypher query templates aligned with Intent enum."""
        return {
            # ================================================================
            # 1. HOTEL_SEARCH
            # ================================================================
            'HOTEL_SEARCH': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                  AND (h.star_rating >= $min_star_rating OR $min_star_rating IS NULL)
                  AND (
                    ($feature_type IS NULL) OR
                    ($feature_type = 'cleanliness' AND h.cleanliness_base >= $min_feature_score) OR
                    ($feature_type = 'comfort' AND h.comfort_base >= $min_feature_score) OR
                    ($feature_type = 'facilities' AND h.facilities_base >= $min_feature_score) OR
                    ($feature_type = 'location' AND h.average_reviews_score >= $min_feature_score) OR
                    ($feature_type = 'staff' AND h.average_reviews_score >= $min_feature_score)
                  )
                RETURN h.hotel_id AS hotel_id,
                       h.name AS hotel_name, 
                       h.star_rating AS star_rating,
                       h.average_reviews_score AS avg_score,
                       h.cleanliness_base AS cleanliness,
                       h.comfort_base AS comfort,
                       h.facilities_base AS facilities,
                       c.name AS city,
                       co.name AS country
                ORDER BY 
                  CASE $feature_type
                    WHEN 'cleanliness' THEN h.cleanliness_base
                    WHEN 'comfort' THEN h.comfort_base
                    WHEN 'facilities' THEN h.facilities_base
                    ELSE h.average_reviews_score
                  END DESC,
                  h.average_reviews_score DESC
                LIMIT $limit
            """,
            
            # ================================================================
            # 2. HOTEL_RECOMMENDATION
            # ================================================================
            'HOTEL_RECOMMENDATION': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                
                OPTIONAL MATCH (t:Traveller)-[:STAYED_AT]->(h)
                WHERE (toLower(t.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                
                WITH h, c, co, COUNT(DISTINCT t) AS similar_traveler_count,
                  CASE $feature_preference
                    WHEN 'cleanliness' THEN h.cleanliness_base
                    WHEN 'comfort' THEN h.comfort_base
                    WHEN 'facilities' THEN h.facilities_base
                    WHEN 'location' THEN h.average_reviews_score
                    WHEN 'staff' THEN h.average_reviews_score
                    WHEN 'value' THEN h.average_reviews_score
                    ELSE h.average_reviews_score
                  END AS feature_score
                
                WHERE feature_score >= $min_score OR $min_score IS NULL
                
                RETURN h.hotel_id AS hotel_id,
                       h.name AS hotel_name,
                       h.star_rating AS star_rating,
                       h.cleanliness_base AS cleanliness,
                       h.comfort_base AS comfort,
                       h.facilities_base AS facilities,
                       h.average_reviews_score AS overall_score,
                       c.name AS city,
                       co.name AS country,
                       similar_traveler_count AS travelers_like_you,
                       feature_score AS recommended_score
                ORDER BY feature_score DESC, similar_traveler_count DESC, h.average_reviews_score DESC
                LIMIT $limit
            """,
            
            # ================================================================
            # 3. REVIEW_QUERY
            # ================================================================
            'REVIEW_QUERY': """
                MATCH (r:Review)-[:REVIEWED]->(h:Hotel)-[:LOCATED_IN]->(c:City)
                MATCH (t:Traveller)-[:WROTE]->(r)
                
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(h.name) CONTAINS toLower($hotel_name) OR $hotel_name IS NULL)
                  AND (r.score_overall >= $min_score OR $min_score IS NULL)
                  AND (r.score_overall <= $max_score OR $max_score IS NULL)
                  AND (toLower(t.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                  AND (t.age >= $age_min OR $age_min IS NULL)
                  AND (t.age <= $age_max OR $age_max IS NULL)
                  AND (toLower(t.gender) = toLower($gender) OR $gender IS NULL)
                
                RETURN h.hotel_id AS hotel_id,
                       h.name AS hotel_name,
                       c.name AS city,
                       r.text AS review_text,
                       r.score_overall AS overall_score,
                       r.score_cleanliness AS cleanliness_score,
                       r.score_comfort AS comfort_score,
                       r.score_facilities AS facilities_score,
                       r.score_location AS location_score,
                       r.score_staff AS staff_score,
                       r.date AS review_date,
                       t.type AS traveler_type,
                       t.age AS traveler_age,
                       t.gender AS traveler_gender
                ORDER BY r.date DESC
                LIMIT $limit
            """,
            
            # ================================================================
            # 4. HOTEL_COMPARISON
            # ================================================================
            'HOTEL_COMPARISON': """
                MATCH (h1:Hotel)-[:LOCATED_IN]->(c1:City)-[:LOCATED_IN]->(co1:Country)
                WHERE (toLower(h1.name) CONTAINS toLower($hotel_name_1) OR 
                       (toLower(c1.name) = toLower($city_1) AND $hotel_name_1 IS NULL))
                WITH h1, c1, co1
                ORDER BY h1.average_reviews_score DESC
                LIMIT 1
                
                MATCH (h2:Hotel)-[:LOCATED_IN]->(c2:City)-[:LOCATED_IN]->(co2:Country)
                WHERE (toLower(h2.name) CONTAINS toLower($hotel_name_2) OR 
                       (toLower(c2.name) = toLower($city_2) AND $hotel_name_2 IS NULL))
                  AND h2.hotel_id <> h1.hotel_id
                WITH h1, c1, co1, h2, c2, co2
                ORDER BY h2.average_reviews_score DESC
                LIMIT 1
                
                OPTIONAL MATCH (t1:Traveller)-[:STAYED_AT]->(h1)
                OPTIONAL MATCH (t2:Traveller)-[:STAYED_AT]->(h2)
                
                WITH h1, c1, co1, h2, c2, co2,
                     COUNT(DISTINCT t1) AS travelers_h1,
                     COUNT(DISTINCT t2) AS travelers_h2
                
                RETURN h1.hotel_id AS hotel_1_id,
                       h1.name AS hotel_1_name,
                       c1.name AS city_1,
                       co1.name AS country_1,
                       h1.star_rating AS hotel_1_stars,
                       h1.average_reviews_score AS hotel_1_rating,
                       h1.cleanliness_base AS hotel_1_cleanliness,
                       h1.comfort_base AS hotel_1_comfort,
                       h1.facilities_base AS hotel_1_facilities,
                       travelers_h1 AS hotel_1_traveler_count,
                       
                       h2.hotel_id AS hotel_2_id,
                       h2.name AS hotel_2_name,
                       c2.name AS city_2,
                       co2.name AS country_2,
                       h2.star_rating AS hotel_2_stars,
                       h2.average_reviews_score AS hotel_2_rating,
                       h2.cleanliness_base AS hotel_2_cleanliness,
                       h2.comfort_base AS hotel_2_comfort,
                       h2.facilities_base AS hotel_2_facilities,
                       travelers_h2 AS hotel_2_traveler_count,
                       
                       (h1.average_reviews_score - h2.average_reviews_score) AS rating_difference,
                       (h1.cleanliness_base - h2.cleanliness_base) AS cleanliness_difference
            """,
            
            # ================================================================
            # 5. VISA_INQUIRY
            # ================================================================
            'VISA_INQUIRY': """
                MATCH (from:Country)
                WHERE toLower(from.name) = toLower($from_country)
                
                MATCH (to:Country)
                WHERE toLower(to.name) = toLower($to_country)
                
                OPTIONAL MATCH (from)-[v:NEEDS_VISA]->(to)
                
                RETURN from.name AS from_country,
                       to.name AS to_country,
                       CASE WHEN v IS NOT NULL 
                            THEN 'Yes' 
                            ELSE 'No' 
                       END AS visa_required,
                       v.visa_type AS visa_type
            """,
            
            # ================================================================
            # 6. LOCATION_BASED_QUERY
            # ================================================================
            'LOCATION_BASED_QUERY': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(co.name) = toLower($country) OR $country IS NULL)
                  AND (h.star_rating >= $min_star_rating OR $min_star_rating IS NULL)
                
                WITH c, co, 
                     COUNT(h) AS hotel_count, 
                     AVG(h.average_reviews_score) AS avg_rating,
                     AVG(h.cleanliness_base) AS avg_cleanliness,
                     AVG(h.comfort_base) AS avg_comfort,
                     AVG(h.facilities_base) AS avg_facilities,
                     MAX(h.star_rating) AS max_stars
                
                WHERE hotel_count >= $min_hotels OR $min_hotels IS NULL
                
                RETURN c.name AS city,
                       co.name AS country,
                       hotel_count AS number_of_hotels,
                       ROUND(avg_rating * 100) / 100 AS average_rating,
                       ROUND(avg_cleanliness * 100) / 100 AS average_cleanliness,
                       ROUND(avg_comfort * 100) / 100 AS average_comfort,
                       ROUND(avg_facilities * 100) / 100 AS average_facilities,
                       max_stars AS highest_star_rating
                ORDER BY hotel_count DESC, avg_rating DESC
                LIMIT $limit
            """,
            
            # ================================================================
            # 7. TRAVELER_PREFERENCE_QUERY
            # ================================================================
            'TRAVELER_PREFERENCE_QUERY': """
                MATCH (t:Traveller)-[:STAYED_AT]->(h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(t.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                  AND (t.age >= $age_min OR $age_min IS NULL)
                  AND (t.age <= $age_max OR $age_max IS NULL)
                  AND (toLower(t.gender) = toLower($gender) OR $gender IS NULL)
                  AND (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                
                WITH h, c, co, t.type AS traveler_type,
                     COUNT(DISTINCT t) AS traveler_count,
                     AVG(t.age) AS avg_age
                
                WHERE traveler_count >= $min_travelers OR $min_travelers IS NULL
                
                RETURN h.hotel_id AS hotel_id,
                       h.name AS hotel_name,
                       h.star_rating AS star_rating,
                       h.average_reviews_score AS avg_score,
                       c.name AS city,
                       co.name AS country,
                       traveler_type AS most_common_traveler_type,
                       traveler_count AS matching_travelers,
                       ROUND(avg_age) AS average_traveler_age
                ORDER BY traveler_count DESC, h.average_reviews_score DESC
                LIMIT $limit
            """,
            
            # ================================================================
            # 8. RATING_ANALYSIS
            # ================================================================
            'RATING_ANALYSIS': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                  AND (h.star_rating = $star_rating OR $star_rating IS NULL)
                
                OPTIONAL MATCH (r:Review)-[:REVIEWED]->(h)
                
                WITH h, c, co,
                     COUNT(DISTINCT r) AS review_count,
                     AVG(r.score_overall) AS avg_review_score,
                     AVG(r.score_cleanliness) AS avg_review_cleanliness,
                     AVG(r.score_comfort) AS avg_review_comfort,
                     AVG(r.score_facilities) AS avg_review_facilities
                
                WHERE review_count >= $min_reviews OR $min_reviews IS NULL
                
                RETURN h.hotel_id AS hotel_id,
                       h.name AS hotel_name,
                       h.star_rating AS star_rating,
                       c.name AS city,
                       co.name AS country,
                       review_count AS total_reviews,
                       ROUND(h.average_reviews_score * 100) / 100 AS base_overall_rating,
                       ROUND(h.cleanliness_base * 100) / 100 AS base_cleanliness,
                       ROUND(h.comfort_base * 100) / 100 AS base_comfort,
                       ROUND(h.facilities_base * 100) / 100 AS base_facilities,
                       ROUND(avg_review_score * 100) / 100 AS review_overall_rating,
                       CASE 
                         WHEN h.cleanliness_base >= h.comfort_base AND h.cleanliness_base >= h.facilities_base
                         THEN 'Cleanliness'
                         WHEN h.comfort_base >= h.facilities_base
                         THEN 'Comfort'
                         ELSE 'Facilities'
                       END AS strongest_aspect
                ORDER BY 
                  CASE $sort_by
                    WHEN 'overall' THEN h.average_reviews_score
                    WHEN 'cleanliness' THEN h.cleanliness_base
                    WHEN 'comfort' THEN h.comfort_base
                    WHEN 'facilities' THEN h.facilities_base
                    WHEN 'reviews' THEN review_count
                    ELSE h.average_reviews_score
                  END DESC
                LIMIT $limit
            """,
            
            # ================================================================
            # 9. STATISTICAL_QUERY
            # ================================================================
            'STATISTICAL_QUERY': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                  AND (h.star_rating = $star_rating OR $star_rating IS NULL)
                
                OPTIONAL MATCH (t:Traveller)-[:STAYED_AT]->(h)
                OPTIONAL MATCH (r:Review)-[:REVIEWED]->(h)
                
                WITH c, co,
                     COUNT(DISTINCT h) AS total_hotels,
                     AVG(h.average_reviews_score) AS avg_overall_rating,
                     AVG(h.cleanliness_base) AS avg_cleanliness,
                     AVG(h.comfort_base) AS avg_comfort,
                     AVG(h.facilities_base) AS avg_facilities,
                     AVG(h.star_rating) AS avg_star_rating,
                     COUNT(DISTINCT r) AS total_reviews,
                     COUNT(DISTINCT t) AS total_travelers
                
                WHERE total_hotels > 0
                
                RETURN c.name AS city,
                       co.name AS country,
                       total_hotels,
                       ROUND(avg_overall_rating * 100) / 100 AS average_rating,
                       ROUND(avg_cleanliness * 100) / 100 AS average_cleanliness,
                       ROUND(avg_comfort * 100) / 100 AS average_comfort,
                       ROUND(avg_facilities * 100) / 100 AS average_facilities,
                       ROUND(avg_star_rating * 10) / 10 AS average_star_rating,
                       total_reviews,
                       total_travelers
                ORDER BY total_hotels DESC, avg_overall_rating DESC
                LIMIT $limit
            """,
            
            # ================================================================
            # 10. DEMOGRAPHIC_RECOMMENDATION
            # ================================================================
            'DEMOGRAPHIC_RECOMMENDATION': """
                MATCH (similar:Traveller)-[:STAYED_AT]->(h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                
                WHERE (similar.age >= $age - $age_range AND similar.age <= $age + $age_range)
                  AND (toLower(similar.gender) = toLower($gender) OR $gender IS NULL)
                  AND (toLower(similar.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                  AND (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                
                OPTIONAL MATCH (similar)-[:WROTE]->(r:Review)-[:REVIEWED]->(h)
                
                WITH h, c, co,
                     COUNT(DISTINCT similar) AS similar_traveler_count,
                     AVG(r.score_overall) AS avg_rating_from_similar,
                     AVG(similar.age) AS avg_similar_age
                
                WHERE similar_traveler_count >= $min_similar_travelers OR $min_similar_travelers IS NULL
                
                RETURN h.hotel_id AS hotel_id,
                       h.name AS hotel_name,
                       h.star_rating AS star_rating,
                       h.average_reviews_score AS overall_avg_score,
                       h.cleanliness_base AS cleanliness,
                       h.comfort_base AS comfort,
                       h.facilities_base AS facilities,
                       c.name AS city,
                       co.name AS country,
                       similar_traveler_count AS travelers_like_you,
                       ROUND(COALESCE(avg_rating_from_similar, h.average_reviews_score) * 100) / 100 AS score_from_similar_travelers,
                       ROUND(avg_similar_age) AS average_age_of_similar_travelers
                ORDER BY similar_traveler_count DESC, score_from_similar_travelers DESC
                LIMIT $limit
            """
        }
    
    def execute_query(self, intent: str, params: Dict[str, Any]) -> List[Dict]:
        """
        Execute the appropriate query with parameters.
        
        Args:
            intent: The intent type (matches query template key)
            params: Dictionary of parameters for the query
            
        Returns:
            List of result dictionaries
        """
        query = self.query_templates.get(intent)
        if not query:
            raise ValueError(f"Unknown intent: {intent}. Available intents: {list(self.query_templates.keys())}")
        
        # Ensure all possible parameters have a value (None if not provided)
        all_possible_params = [
            'city', 'country', 'hotel_name', 'min_star_rating', 'star_rating',
            'feature_type', 'min_feature_score', 'feature_preference', 'min_score', 'max_score',
            'traveler_type', 'age', 'age_min', 'age_max', 'age_range', 'gender',
            'limit', 'min_hotels', 'min_travelers', 'min_similar_travelers', 'min_reviews',
            'from_country', 'to_country',
            'hotel_name_1', 'hotel_name_2', 'city_1', 'city_2',
            'sort_by'
        ]
        
        for key in all_possible_params:
            if key not in params:
                params[key] = None
        
        try:
            with self.driver.session() as session:
                result = session.run(query, **params)
                return [dict(record) for record in result]
        except Exception as e:
            print(f"Error executing query for intent '{intent}': {str(e)}")
            print(f"Parameters: {params}")
            raise
    
    def close(self):
        """Close the database connection."""
        self.driver.close()

In [37]:
# Initialize components
mapper = QueryMapper()
executor = QueryExecutor(
    uri="neo4j://127.0.0.1:7687",
    user="neo4j",
    password="Omarino2003"
)

try:
    # Example 1: Basic hotel search
    entities_1 = {
        'city': ['Paris'],
        'star_rating': ['5']
    }
    intent_1 = 'HOTEL_SEARCH'  # ✅ Changed from 'HOTEL_SEARCH_BASIC'
    params_1 = mapper.map_entities_to_parameters(intent_1, entities_1)
    results_1 = executor.execute_query(intent_1, params_1)
    print(f"Found {len(results_1)} hotels")
    print(results_1)
    
    # Example 2: Demographic recommendation
    entities_2 = {
        'age': ['28'],
        'gender': ['female'],
        'traveler_type': ['solo'],
        'city': ['Paris']
    }
    intent_2 = 'DEMOGRAPHIC_RECOMMENDATION'  # ✅ This one was already correct!
    params_2 = mapper.map_entities_to_parameters(intent_2, entities_2)
    results_2 = executor.execute_query(intent_2, params_2)
    print(f"Found {len(results_2)} recommendations")
    print(results_2)
    
    # Example 3: Visa inquiry
    entities_3 = {
        'country': ['Egypt', 'France']
    }
    intent_3 = 'VISA_INQUIRY'  # ✅ This one was already correct!
    params_3 = mapper.map_entities_to_parameters(intent_3, entities_3)
    results_3 = executor.execute_query(intent_3, params_3)
    print(f"Visa info: {results_3}")
    
finally:
    executor.close()

Found 1 hotels
[{'hotel_id': '3', 'hotel_name': "L'Étoile Palace", 'star_rating': 5.0, 'avg_score': 8.9470647265429, 'cleanliness': 8.8, 'comfort': 9.4, 'facilities': 8.7, 'city': 'Paris', 'country': 'France'}]
Found 0 recommendations
[]
Visa info: [{'from_country': 'Egypt', 'to_country': 'France', 'visa_required': 'Yes', 'visa_type': 'Tourist Visa'}]


NOT HARD-CODED: Entities are called using the entity extractor method and the intents are done using the LLM intent classifier above.

In [38]:
# Initialize components
entity_extractor = EntityExtractor()  # Your entity extractor from earlier
mapper = QueryMapper()
executor = QueryExecutor(
    uri="neo4j://127.0.0.1:7687",
    user="neo4j",
    password="Omarino2003"
)

try:
    # Example 1: Basic hotel search
    query_1 = "Find 5-star hotels in Paris"
    print("="*80)
    print(f"Query 1: {query_1}")
    print("="*80)
    
    # Extract entities and classify intent
    entities_1 = entity_extractor.extract_entities(query_1)
    intent_1 = classify_hotel_intent(query_1)  # Your LLM classifier
    
    print(f"Classified Intent: {intent_1}")
    print(f"Extracted Entities: {entities_1}")
    
    # Map and execute
    params_1 = mapper.map_entities_to_parameters(intent_1, entities_1)
    results_1 = executor.execute_query(intent_1, params_1)
    print(f"\nFound {len(results_1)} hotels")
    for hotel in results_1[:3]:  # Show top 3
        print(f"  - {hotel['hotel_name']}: {hotel['avg_score']:.1f}/10, {hotel['star_rating']} stars")
    print()
    
    # Example 2: Demographic recommendation
    query_2 = "Recommend a hotel for a 28-year-old female solo traveler in Paris"
    print("="*80)
    print(f"Query 2: {query_2}")
    print("="*80)
    
    # Extract entities and classify intent
    entities_2 = entity_extractor.extract_entities(query_2)
    intent_2 = classify_hotel_intent(query_2)
    
    print(f"Classified Intent: {intent_2}")
    print(f"Extracted Entities: {entities_2}")
    
    # Map and execute
    params_2 = mapper.map_entities_to_parameters(intent_2, entities_2)
    results_2 = executor.execute_query(intent_2, params_2)
    print(f"\nFound {len(results_2)} recommendations")
    for hotel in results_2[:3]:  # Show top 3
        print(f"  - {hotel['hotel_name']}: {hotel['travelers_like_you']} similar travelers")
    print()
    
    # Example 3: Visa inquiry
    query_3 = "Do I need a visa to travel from Egypt to France?"
    print("="*80)
    print(f"Query 3: {query_3}")
    print("="*80)
    
    # Extract entities and classify intent
    entities_3 = entity_extractor.extract_entities(query_3)
    intent_3 = classify_hotel_intent(query_3)
    
    print(f"Classified Intent: {intent_3}")
    print(f"Extracted Entities: {entities_3}")
    
    # Map and execute
    params_3 = mapper.map_entities_to_parameters(intent_3, entities_3)
    results_3 = executor.execute_query(intent_3, params_3)
    print(f"\nVisa requirement: {results_3}")
    print()
    
    # Example 4: Review query
    query_4 = "What do people say about hotels in Dubai?"
    print("="*80)
    print(f"Query 4: {query_4}")
    print("="*80)
    
    entities_4 = entity_extractor.extract_entities(query_4)
    intent_4 = classify_hotel_intent(query_4)
    
    print(f"Classified Intent: {intent_4}")
    print(f"Extracted Entities: {entities_4}")
    
    params_4 = mapper.map_entities_to_parameters(intent_4, entities_4)
    results_4 = executor.execute_query(intent_4, params_4)
    print(f"\nFound {len(results_4)} reviews")
    for review in results_4[:2]:  # Show top 2
        print(f"  - {review['hotel_name']}: {review['overall_score']}/10 by {review['traveler_type']}")
    print()
    
    # Example 5: Hotel comparison
    query_5 = "Compare hotels in Paris vs London"
    print("="*80)
    print(f"Query 5: {query_5}")
    print("="*80)
    
    entities_5 = entity_extractor.extract_entities(query_5)
    intent_5 = classify_hotel_intent(query_5)
    
    print(f"Classified Intent: {intent_5}")
    print(f"Extracted Entities: {entities_5}")
    
    params_5 = mapper.map_entities_to_parameters(intent_5, entities_5)
    results_5 = executor.execute_query(intent_5, params_5)
    
    if results_5:
        result = results_5[0]
        print(f"\nComparison:")
        print(f"  Hotel 1: {result['hotel_1_name']} ({result['city_1']})")
        print(f"    Rating: {result['hotel_1_rating']:.1f}/10")
        print(f"  Hotel 2: {result['hotel_2_name']} ({result['city_2']})")
        print(f"    Rating: {result['hotel_2_rating']:.1f}/10")
        print(f"  Difference: {result['rating_difference']:.2f}")
    print()
    
finally:
    executor.close()

Query 1: Find 5-star hotels in Paris
Classified Intent: HOTEL_SEARCH
Extracted Entities: {'city': ['paris'], 'star_rating': ['5']}

Found 1 hotels
  - L'Étoile Palace: 8.9/10, 5.0 stars

Query 2: Recommend a hotel for a 28-year-old female solo traveler in Paris
Classified Intent: HOTEL_RECOMMENDATION
Extracted Entities: {'city': ['paris'], 'traveler_type': ['solo', 'solo traveler'], 'gender': ['female']}

Found 1 recommendations
  - L'Étoile Palace: 245 similar travelers

Query 3: Do I need a visa to travel from Egypt to France?
Classified Intent: VISA_INQUIRY
Extracted Entities: {'country': ['egypt', 'france']}

Visa requirement: [{'from_country': 'Egypt', 'to_country': 'France', 'visa_required': 'Yes', 'visa_type': 'Tourist Visa'}]

Query 4: What do people say about hotels in Dubai?
Classified Intent: REVIEW_QUERY
Extracted Entities: {'city': ['dubai']}

Found 20 reviews
  - The Golden Oasis: 9.2/10 by Couple
  - The Golden Oasis: 9.1/10 by Business

Query 5: Compare hotels in Paris 

EMBEDDINGS

In [39]:
class EnhancedFeatureVectorEmbedder:
    """
    Create feature vector embeddings for:
    1. Hotels (with review text AND scores)
    2. Visa requirements (country pairs with visa info)
    """
    
    def __init__(self, neo4j_uri: str, neo4j_user: str, neo4j_password: str):
        """Initialize with Neo4j connection and embedding models."""
        # Connect to Neo4j
        self.driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
        
        # Load both embedding models
        print("Loading SBERT model...")
        self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        print("Loading BGE model...")
        self.bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5')
        
        print("Models loaded successfully!")
    
    # ========================================================================
    # HOTEL EMBEDDINGS (Enhanced with Review Scores)
    # ========================================================================
    
    def fetch_hotels_with_features(self) -> List[Dict[str, Any]]:
        """
        Fetch all hotels with their properties, reviews, AND review scores.
        """
        query = """
        MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
        OPTIONAL MATCH (r:Review)-[:REVIEWED]->(h)
        OPTIONAL MATCH (t:Traveller)-[:STAYED_AT]->(h)
        
        WITH h, c, co,
             COLLECT(DISTINCT r.text)[0..5] AS sample_reviews,
             COLLECT(DISTINCT {
                 overall: r.score_overall,
                 cleanliness: r.score_cleanliness,
                 comfort: r.score_comfort,
                 facilities: r.score_facilities,
                 location: r.score_location,
                 staff: r.score_staff,
                 value_for_money: r.score_value_for_money
             })[0..5] AS review_scores,
             COUNT(DISTINCT r) AS review_count,
             COUNT(DISTINCT t) AS traveler_count,
             COLLECT(DISTINCT t.type) AS traveler_types
        
        RETURN h.hotel_id AS hotel_id,
               h.name AS hotel_name,
               h.star_rating AS star_rating,
               h.average_reviews_score AS avg_score,
               h.cleanliness_base AS cleanliness,
               h.comfort_base AS comfort,
               h.facilities_base AS facilities,
               c.name AS city,
               co.name AS country,
               sample_reviews,
               review_scores,
               review_count,
               traveler_count,
               traveler_types
        """
        
        with self.driver.session() as session:
            result = session.run(query)
            hotels = [dict(record) for record in result]
        
        print(f"Fetched {len(hotels)} hotels from Neo4j")
        return hotels
    
    def create_hotel_feature_text(self, hotel: Dict[str, Any]) -> str:
        """
        Combine hotel properties into rich text representation.
        NOW INCLUDES: Review scores alongside review text!
        
        Args:
            hotel: Dictionary containing hotel properties
            
        Returns:
            Combined text representation
        """
        parts = []
        
        # Basic info
        parts.append(f"Hotel: {hotel['hotel_name']}")
        parts.append(f"Location: {hotel['city']}, {hotel['country']}")
        parts.append(f"Star rating: {hotel['star_rating']} stars")
        
        # Quality scores (hotel base scores)
        if hotel['avg_score']:
            parts.append(f"Overall rating: {hotel['avg_score']:.1f}/5")
        if hotel['cleanliness']:
            parts.append(f"Cleanliness: {hotel['cleanliness']:.1f}/5")
        if hotel['comfort']:
            parts.append(f"Comfort: {hotel['comfort']:.1f}/5")
        if hotel['facilities']:
            parts.append(f"Facilities: {hotel['facilities']:.1f}/5")
        
        # Traveler information
        if hotel['traveler_count'] and hotel['traveler_count'] > 0:
            parts.append(f"Popular with {hotel['traveler_count']} travelers")
        
        if hotel['traveler_types']:
            traveler_types_str = ', '.join([t for t in hotel['traveler_types'] if t])
            if traveler_types_str:
                parts.append(f"Traveler types: {traveler_types_str}")
        
        # ====================================================================
        # ENHANCED: Review scores + review text together!
        # ====================================================================
        
        if hotel['review_scores']:
            # Calculate average scores across sample reviews
            valid_scores = [s for s in hotel['review_scores'] if s and any(s.values())]
            
            if valid_scores:
                # Aggregate scores
                avg_scores = {
                    'overall': np.mean([s['overall'] for s in valid_scores if s.get('overall')]),
                    'cleanliness': np.mean([s['cleanliness'] for s in valid_scores if s.get('cleanliness')]),
                    'comfort': np.mean([s['comfort'] for s in valid_scores if s.get('comfort')]),
                    'facilities': np.mean([s['facilities'] for s in valid_scores if s.get('facilities')]),
                    'location': np.mean([s['location'] for s in valid_scores if s.get('location')]),
                    'staff': np.mean([s['staff'] for s in valid_scores if s.get('staff')]),
                    'value_for_money': np.mean([s['value_for_money'] for s in valid_scores if s.get('value_for_money')])
                }
                
                # Add detailed score breakdown
                score_parts = []
                if not np.isnan(avg_scores['overall']):
                    score_parts.append(f"guest overall: {avg_scores['overall']:.1f}/5")
                if not np.isnan(avg_scores['cleanliness']):
                    score_parts.append(f"guest cleanliness: {avg_scores['cleanliness']:.1f}/5")
                if not np.isnan(avg_scores['comfort']):
                    score_parts.append(f"guest comfort: {avg_scores['comfort']:.1f}/5")
                if not np.isnan(avg_scores['facilities']):
                    score_parts.append(f"guest facilities: {avg_scores['facilities']:.1f}/5")
                if not np.isnan(avg_scores['location']):
                    score_parts.append(f"guest location: {avg_scores['location']:.1f}/5")
                if not np.isnan(avg_scores['staff']):
                    score_parts.append(f"guest staff: {avg_scores['staff']:.1f}/5")
                if not np.isnan(avg_scores['value_for_money']):
                    score_parts.append(f"guest value: {avg_scores['value_for_money']:.1f}/5")
                
                if score_parts:
                    parts.append(f"Guest ratings: {', '.join(score_parts)}")
        
        # Sample reviews (textual content)
        if hotel['sample_reviews']:
            valid_reviews = [r for r in hotel['sample_reviews'] if r and len(r.strip()) > 10]
            if valid_reviews:
                review_text = ' '.join(valid_reviews[:3])
                parts.append(f"Guest reviews: {review_text}")
        
        # Combine all parts
        feature_text = '. '.join(parts)
        
        return feature_text
    
    # ========================================================================
    # VISA EMBEDDINGS
    # ========================================================================
    
    def fetch_visa_requirements(self) -> List[Dict[str, Any]]:
        """
        Fetch all visa requirement relationships from Neo4j.
        Each visa requirement is a (Country)-[:NEEDS_VISA]->(Country) relationship.
        
        Returns:
            List of dictionaries containing visa requirement data
        """
        query = """
        MATCH (from:Country)-[v:NEEDS_VISA]->(to:Country)
        RETURN from.name AS from_country,
               to.name AS to_country,
               v.visa_type AS visa_type
        """
        
        with self.driver.session() as session:
            result = session.run(query)
            visa_reqs = [dict(record) for record in result]
        
        print(f"Fetched {len(visa_reqs)} visa requirements from Neo4j")
        return visa_reqs
    
    def create_visa_feature_text(self, visa_req: Dict[str, Any]) -> str:
        """
        Create text representation of a visa requirement.
        
        Args:
            visa_req: Dictionary with from_country, to_country, visa_type
            
        Returns:
            Text representation of visa requirement
        """
        from_country = visa_req['from_country']
        to_country = visa_req['to_country']
        visa_type = visa_req.get('visa_type', 'required')
        
        # Create rich textual representation
        text = (
            f"Visa requirement: Travelers from {from_country} need a visa to enter {to_country}. "
            f"Visa type: {visa_type}. "
            f"Travel from {from_country} to {to_country} requires {visa_type} visa. "
            f"{from_country} citizens traveling to {to_country} must obtain visa."
        )
        
        return text
    
    def create_visa_id(self, from_country: str, to_country: str) -> str:
        """
        Create a unique ID for a visa requirement.
        
        Args:
            from_country: Source country
            to_country: Destination country
            
        Returns:
            Unique ID string
        """
        # Normalize country names and create ID
        from_normalized = from_country.lower().replace(' ', '_')
        to_normalized = to_country.lower().replace(' ', '_')
        return f"visa_{from_normalized}_to_{to_normalized}"
    
    # ========================================================================
    # EMBEDDING GENERATION
    # ========================================================================
    
    def embed_hotels(self, hotels: List[Dict[str, Any]], model_name: str = 'both') -> Dict[str, Any]:
        """
        Create embeddings for all hotels using specified model(s).
        
        Args:
            hotels: List of hotel dictionaries
            model_name: 'sbert', 'bge', or 'both'
            
        Returns:
            Dictionary mapping hotel_ids to their embeddings
        """
        embeddings_data = {
            'sbert': {},
            'bge': {}
        }
        
        print(f"\nCreating hotel embeddings for {len(hotels)} hotels...")
        
        for hotel in tqdm(hotels, desc="Embedding hotels"):
            # Create enhanced feature text (with review scores!)
            feature_text = self.create_hotel_feature_text(hotel)
            hotel_id = hotel['hotel_id']
            
            # Create embeddings with both models
            if model_name in ['sbert', 'both']:
                sbert_embedding = self.sbert_model.encode(feature_text)
                embeddings_data['sbert'][hotel_id] = sbert_embedding.tolist()
            
            if model_name in ['bge', 'both']:
                bge_embedding = self.bge_model.encode(feature_text)
                embeddings_data['bge'][hotel_id] = bge_embedding.tolist()
        
        print(f"✓ Created hotel embeddings for {len(hotels)} hotels")
        return embeddings_data
    
    def embed_visa_requirements(self, visa_reqs: List[Dict[str, Any]], model_name: str = 'both') -> Dict[str, Any]:
        """
        Create embeddings for all visa requirements using specified model(s).
        
        Args:
            visa_reqs: List of visa requirement dictionaries
            model_name: 'sbert', 'bge', or 'both'
            
        Returns:
            Dictionary mapping visa_ids to their embeddings
        """
        embeddings_data = {
            'sbert': {},
            'bge': {}
        }
        
        print(f"\nCreating visa embeddings for {len(visa_reqs)} visa requirements...")
        
        for visa_req in tqdm(visa_reqs, desc="Embedding visa requirements"):
            # Create feature text
            feature_text = self.create_visa_feature_text(visa_req)
            visa_id = self.create_visa_id(visa_req['from_country'], visa_req['to_country'])
            
            # Create embeddings with both models
            if model_name in ['sbert', 'both']:
                sbert_embedding = self.sbert_model.encode(feature_text)
                embeddings_data['sbert'][visa_id] = sbert_embedding.tolist()
            
            if model_name in ['bge', 'both']:
                bge_embedding = self.bge_model.encode(feature_text)
                embeddings_data['bge'][visa_id] = bge_embedding.tolist()
        
        print(f"✓ Created visa embeddings for {len(visa_reqs)} visa requirements")
        return embeddings_data
    
    # ========================================================================
    # STORAGE IN NEO4J
    # ========================================================================
    
    def store_hotel_embeddings_in_neo4j(self, embeddings_data: Dict[str, Any]):
        """
        Store hotel embeddings in Neo4j as properties on Hotel nodes.
        
        Args:
            embeddings_data: Dictionary with 'sbert' and 'bge' embeddings
        """
        print("\nStoring hotel embeddings in Neo4j...")
        
        # Store SBERT embeddings
        if embeddings_data['sbert']:
            print("Storing hotel SBERT embeddings...")
            with self.driver.session() as session:
                for hotel_id, embedding in tqdm(embeddings_data['sbert'].items(), 
                                                desc="Hotel SBERT"):
                    session.run("""
                        MATCH (h:Hotel {hotel_id: $hotel_id})
                        SET h.embedding_sbert = $embedding
                    """, hotel_id=hotel_id, embedding=embedding)
        
        # Store BGE embeddings
        if embeddings_data['bge']:
            print("Storing hotel BGE embeddings...")
            with self.driver.session() as session:
                for hotel_id, embedding in tqdm(embeddings_data['bge'].items(),
                                                desc="Hotel BGE"):
                    session.run("""
                        MATCH (h:Hotel {hotel_id: $hotel_id})
                        SET h.embedding_bge = $embedding
                    """, hotel_id=hotel_id, embedding=embedding)
        
        print("✓ All hotel embeddings stored in Neo4j")
    
    def store_visa_embeddings_in_neo4j(self, embeddings_data: Dict[str, Any], visa_reqs: List[Dict[str, Any]]):
        """
        Store visa embeddings in Neo4j as properties on NEEDS_VISA relationships.
        
        Args:
            embeddings_data: Dictionary with 'sbert' and 'bge' embeddings
            visa_reqs: Original visa requirement data (for from/to countries)
        """
        print("\nStoring visa embeddings in Neo4j...")
        
        # Create a mapping from visa_id to from/to countries
        visa_id_to_countries = {
            self.create_visa_id(v['from_country'], v['to_country']): v
            for v in visa_reqs
        }
        
        # Store SBERT embeddings
        if embeddings_data['sbert']:
            print("Storing visa SBERT embeddings...")
            with self.driver.session() as session:
                for visa_id, embedding in tqdm(embeddings_data['sbert'].items(),
                                               desc="Visa SBERT"):
                    countries = visa_id_to_countries[visa_id]
                    session.run("""
                        MATCH (from:Country {name: $from_country})-[v:NEEDS_VISA]->(to:Country {name: $to_country})
                        SET v.embedding_sbert = $embedding
                    """, from_country=countries['from_country'],
                        to_country=countries['to_country'],
                        embedding=embedding)
        
        # Store BGE embeddings
        if embeddings_data['bge']:
            print("Storing visa BGE embeddings...")
            with self.driver.session() as session:
                for visa_id, embedding in tqdm(embeddings_data['bge'].items(),
                                               desc="Visa BGE"):
                    countries = visa_id_to_countries[visa_id]
                    session.run("""
                        MATCH (from:Country {name: $from_country})-[v:NEEDS_VISA]->(to:Country {name: $to_country})
                        SET v.embedding_bge = $embedding
                    """, from_country=countries['from_country'],
                        to_country=countries['to_country'],
                        embedding=embedding)
        
        print("✓ All visa embeddings stored in Neo4j")
    
    # ========================================================================
    # VECTOR INDEXES
    # ========================================================================
    
    def create_vector_indexes(self):
        """
        Create vector indexes for both hotels and visa requirements.
        """
        print("\nCreating vector indexes...")
        
        with self.driver.session() as session:
            # Hotel SBERT index
            try:
                session.run("""
                    CREATE VECTOR INDEX hotel_sbert_index IF NOT EXISTS
                    FOR (h:Hotel)
                    ON h.embedding_sbert
                    OPTIONS {
                        indexConfig: {
                            `vector.dimensions`: 384,
                            `vector.similarity_function`: 'cosine'
                        }
                    }
                """)
                print("✓ Created hotel SBERT vector index (384 dimensions)")
            except Exception as e:
                print(f"Hotel SBERT index may already exist: {e}")
            
            # Hotel BGE index
            try:
                session.run("""
                    CREATE VECTOR INDEX hotel_bge_index IF NOT EXISTS
                    FOR (h:Hotel)
                    ON h.embedding_bge
                    OPTIONS {
                        indexConfig: {
                            `vector.dimensions`: 1024,
                            `vector.similarity_function`: 'cosine'
                        }
                    }
                """)
                print("✓ Created hotel BGE vector index (1024 dimensions)")
            except Exception as e:
                print(f"Hotel BGE index may already exist: {e}")
            
            # Note: Neo4j doesn't support vector indexes on relationships yet (as of Neo4j 5.x)
            # So visa embeddings will be stored but searched manually or via FAISS
            print("Note: Visa embeddings stored on relationships (no vector index on relationships yet)")
    
    # ========================================================================
    # VERIFICATION
    # ========================================================================
    
    def verify_embeddings(self, sample_size: int = 5):
        """Verify that embeddings were stored correctly."""
        print(f"\nVerifying embeddings...")
        
        # Check hotels
        print("\n--- Hotel Embeddings ---")
        hotel_query = """
        MATCH (h:Hotel)
        WHERE h.embedding_sbert IS NOT NULL AND h.embedding_bge IS NOT NULL
        RETURN h.hotel_id AS hotel_id,
               h.name AS hotel_name,
               size(h.embedding_sbert) AS sbert_dims,
               size(h.embedding_bge) AS bge_dims
        LIMIT $limit
        """
        
        with self.driver.session() as session:
            result = session.run(hotel_query, limit=sample_size)
            
            for record in result:
                print(f"✓ {record['hotel_name']}: "
                      f"SBERT={record['sbert_dims']}D, "
                      f"BGE={record['bge_dims']}D")
        
        # Count total hotels
        count_query = """
        MATCH (h:Hotel)
        WHERE h.embedding_sbert IS NOT NULL
        RETURN count(h) AS total
        """
        
        with self.driver.session() as session:
            result = session.run(count_query)
            total = result.single()['total']
            print(f"\n✓ Total hotels with embeddings: {total}")
        
        # Check visa requirements
        print("\n--- Visa Embeddings ---")
        visa_query = """
        MATCH (from:Country)-[v:NEEDS_VISA]->(to:Country)
        WHERE v.embedding_sbert IS NOT NULL AND v.embedding_bge IS NOT NULL
        RETURN from.name AS from_country,
               to.name AS to_country,
               size(v.embedding_sbert) AS sbert_dims,
               size(v.embedding_bge) AS bge_dims
        LIMIT $limit
        """
        
        with self.driver.session() as session:
            result = session.run(visa_query, limit=sample_size)
            
            for record in result:
                print(f"✓ {record['from_country']} → {record['to_country']}: "
                      f"SBERT={record['sbert_dims']}D, "
                      f"BGE={record['bge_dims']}D")
        
        # Count total visa requirements
        visa_count_query = """
        MATCH ()-[v:NEEDS_VISA]->()
        WHERE v.embedding_sbert IS NOT NULL
        RETURN count(v) AS total
        """
        
        with self.driver.session() as session:
            result = session.run(visa_count_query)
            total = result.single()['total']
            print(f"\n✓ Total visa requirements with embeddings: {total}")
    
    def close(self):
        """Close Neo4j connection."""
        self.driver.close()


# ============================================================================
# Main Execution
# ============================================================================

def main():
    """
    Main function to create and store ALL embeddings.
    """
    # Configuration
    NEO4J_URI = "neo4j://127.0.0.1:7687"
    NEO4J_USER = "neo4j"
    NEO4J_PASSWORD = "Omarino2003"
    
    # Initialize embedder
    embedder = EnhancedFeatureVectorEmbedder(
        neo4j_uri=NEO4J_URI,
        neo4j_user=NEO4J_USER,
        neo4j_password=NEO4J_PASSWORD
    )
    
    try:
        # ====================================================================
        # PART 1: HOTEL EMBEDDINGS (with review scores!)
        # ====================================================================
        print("\n" + "="*80)
        print("PART 1: CREATING HOTEL EMBEDDINGS")
        print("="*80)
        
        # Fetch hotels
        hotels = embedder.fetch_hotels_with_features()
        
        # Create embeddings
        hotel_embeddings = embedder.embed_hotels(hotels, model_name='both')
        
        # Store in Neo4j
        embedder.store_hotel_embeddings_in_neo4j(hotel_embeddings)
        
        # ====================================================================
        # PART 2: VISA EMBEDDINGS (new!)
        # ====================================================================
        print("\n" + "="*80)
        print("PART 2: CREATING VISA EMBEDDINGS")
        print("="*80)
        
        # Fetch visa requirements
        visa_reqs = embedder.fetch_visa_requirements()
        
        # Create embeddings
        visa_embeddings = embedder.embed_visa_requirements(visa_reqs, model_name='both')
        
        # Store in Neo4j
        embedder.store_visa_embeddings_in_neo4j(visa_embeddings, visa_reqs)
        
        # ====================================================================
        # PART 3: CREATE INDEXES & VERIFY
        # ====================================================================
        print("\n" + "="*80)
        print("PART 3: CREATING INDEXES & VERIFICATION")
        print("="*80)
        
        # Create vector indexes
        embedder.create_vector_indexes()
        
        # Verify all embeddings
        embedder.verify_embeddings(sample_size=5)
        
        print("\n" + "="*80)
        print("SUCCESS! All embeddings created and stored!")
        print("="*80)
        print("\nCreated embeddings for:")
        print("✓ Hotels (with review text AND scores)")
        print("✓ Visa requirements (country pairs)")
        print("\nYou can now use semantic similarity search for both!")
        
    finally:
        embedder.close()


if __name__ == "__main__":
    main()

Loading SBERT model...
Loading BGE model...
Models loaded successfully!

PART 1: CREATING HOTEL EMBEDDINGS
Fetched 25 hotels from Neo4j

Creating hotel embeddings for 25 hotels...


Embedding hotels: 100%|██████████| 25/25 [00:11<00:00,  2.17it/s]


✓ Created hotel embeddings for 25 hotels

Storing hotel embeddings in Neo4j...
Storing hotel SBERT embeddings...


Hotel SBERT: 100%|██████████| 25/25 [00:03<00:00,  6.75it/s]


Storing hotel BGE embeddings...


...

✓ Created visa embeddings for 114 visa requirements

Storing visa embeddings in Neo4j...
Storing visa SBERT embeddings...


Visa SBERT: 100%|██████████| 114/114 [00:01<00:00, 62.11it/s] 


Storing visa BGE embeddings...


Visa BGE: 100%|██████████| 114/114 [00:01<00:00, 81.22it/s] 


✓ All visa embeddings stored in Neo4j

PART 3: CREATING INDEXES & VERIFICATION

Creating vector indexes...
✓ Created hotel SBERT vector index (384 dimensions)
✓ Created hotel BGE vector index (1024 dimensions)
Note: Visa embeddings stored on relationships (no vector index on relationships yet)

Verifying embeddings...

--- Hotel Embeddings ---
✓ The Azure Tower: SBERT=384D, BGE=1024D
✓ The Royal Compass: SBERT=384D, BGE=1024D
✓ L'Étoile Palace: SBERT=384D, BGE=1024D
✓ Kyo-to Grand: SBERT=384D, BGE=1024D
✓ The Golden Oasis: SBERT=384D, BGE=1024D

✓ Total hotels with embeddings: 25

--- Visa Embeddings ---
✓ South Africa → Singapore: SBERT=384D, BGE=1024D
✓ South Africa → Russia: SBERT=384D, BGE=1024D
✓ South Africa → France: SBERT=384D, BGE=1024D
✓ South Africa → Spain: SBERT=384D, BGE=1024D
✓ South Africa → Germany: SBERT=384D, BGE=1024D

✓ Total visa requirements with embeddings: 114

SUCCESS! All embeddings created and stored!

Created embeddings for:
✓ Hotels (with review text AND s

In [102]:
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
import numpy as np
from typing import List, Dict, Tuple, Any
import faiss
import pickle
import os


class EnhancedSemanticSearch:
    """
    Perform semantic similarity search for:
    1. Hotels (using Neo4j vector index)
    2. Visa requirements (using FAISS since Neo4j can't index relationships)
    """

    def __init__(self, neo4j_uri: str, neo4j_user: str, neo4j_password: str):
        """Initialize with Neo4j connection and embedding models."""
        # Connect to Neo4j
        self.driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

        # Load embedding models
        print("Loading embedding models...")
        self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5')

        # FAISS indexes for visa requirements
        self.visa_faiss_index_sbert = None
        self.visa_faiss_index_bge = None
        self.visa_id_to_data = {}  # Maps index position to visa data
        self.index_to_visa_id = {}  # Maps FAISS index to visa_id

        print("✓ Initialization complete")

    # ========================================================================
    # HOTEL SEMANTIC SEARCH (Neo4j Vector Index)
    # ========================================================================

    def search_hotels_semantic(
        self,
        query: str,
        model: str = 'sbert',
        top_k: int = 10,
        city_filter: str = None,  # ADD THIS PARAMETER
        country_filter: str = None  # ADD THIS PARAMETER
    ) -> List[Dict]:
        """
        Search hotels using Neo4j's native vector index.

        Args:
            query: User search query
            model: 'sbert' or 'bge'
            top_k: Number of results to return
            city_filter: Optional city name to filter results
            country_filter: Optional country name to filter results

        Returns:
            List of hotel dictionaries with similarity scores
        """
        # Step 1: Embed the query
        if model == 'sbert':
            query_embedding = self.sbert_model.encode(query).tolist()
            index_name = 'hotel_sbert_index'
        elif model == 'bge':
            query_embedding = self.bge_model.encode(query).tolist()
            index_name = 'hotel_bge_index'
        else:
            raise ValueError("model must be 'sbert' or 'bge'")

        # Step 2: Build WHERE clause for filters
        where_clauses = []
        if city_filter:
            where_clauses.append("toLower(c.name) = toLower($city_filter)")
        if country_filter:
            where_clauses.append("toLower(co.name) = toLower($country_filter)")
        
        where_clause = ""
        if where_clauses:
            where_clause = "WHERE " + " AND ".join(where_clauses)

        # Step 3: Query Neo4j vector index WITH FILTERS
        cypher_query = f"""
        CALL db.index.vector.queryNodes(
            $index_name,
            $top_k * 3,
            $query_embedding
        )
        YIELD node, score
        MATCH (node)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
        {where_clause}
        RETURN node.hotel_id AS hotel_id,
            node.name AS hotel_name,
            node.star_rating AS star_rating,
            node.average_reviews_score AS avg_score,
            node.cleanliness_base AS cleanliness,
            node.comfort_base AS comfort,
            node.facilities_base AS facilities,
            c.name AS city,
            co.name AS country,
            score AS similarity_score
        ORDER BY score DESC
        LIMIT $top_k
        """

        with self.driver.session() as session:
            result = session.run(
                cypher_query,
                index_name=index_name,
                top_k=top_k,
                query_embedding=query_embedding,
                city_filter=city_filter,
                country_filter=country_filter
            )
            hotels = [dict(record) for record in result]

        return hotels

    # ========================================================================
    # VISA SEMANTIC SEARCH (FAISS - since Neo4j can't index relationships)
    # ========================================================================

    def build_visa_faiss_index(self, model: str = 'both'):
        """
        Build FAISS index for visa requirements from Neo4j.
        Must be called once before searching.

        Args:
            model: 'sbert', 'bge', or 'both'
        """
        print("\nBuilding FAISS index for visa requirements...")

        # Fetch all visa embeddings from Neo4j
        query = """
        MATCH (from:Country)-[v:NEEDS_VISA]->(to:Country)
        WHERE v.embedding_sbert IS NOT NULL
        RETURN from.name AS from_country,
               to.name AS to_country,
               v.visa_type AS visa_type,
               v.embedding_sbert AS embedding_sbert,
               v.embedding_bge AS embedding_bge
        """

        with self.driver.session() as session:
            result = session.run(query)
            visa_data = [dict(record) for record in result]

        if not visa_data:
            print("⚠ No visa embeddings found in Neo4j!")
            return

        print(f"Found {len(visa_data)} visa requirements")

        # Prepare embeddings and metadata
        sbert_embeddings = []
        bge_embeddings = []

        for idx, visa in enumerate(visa_data):
            # Create visa_id
            visa_id = f"visa_{visa['from_country'].lower().replace(' ', '_')}_to_{visa['to_country'].lower().replace(' ', '_')}"

            # Store mapping
            self.visa_id_to_data[idx] = {
                'visa_id': visa_id,
                'from_country': visa['from_country'],
                'to_country': visa['to_country'],
                'visa_type': visa['visa_type']
            }
            self.index_to_visa_id[idx] = visa_id

            # Collect embeddings
            if model in ['sbert', 'both'] and visa['embedding_sbert']:
                sbert_embeddings.append(visa['embedding_sbert'])

            if model in ['bge', 'both'] and visa['embedding_bge']:
                bge_embeddings.append(visa['embedding_bge'])

        # Build FAISS indexes
        if model in ['sbert', 'both'] and sbert_embeddings:
            sbert_array = np.array(sbert_embeddings, dtype='float32')
            dimension = sbert_array.shape[1]
            self.visa_faiss_index_sbert = faiss.IndexFlatIP(dimension)  # Inner product (cosine similarity)
            faiss.normalize_L2(sbert_array)  # Normalize for cosine similarity
            self.visa_faiss_index_sbert.add(sbert_array)
            print(f"✓ Built SBERT FAISS index ({dimension}D, {len(sbert_embeddings)} vectors)")

        if model in ['bge', 'both'] and bge_embeddings:
            bge_array = np.array(bge_embeddings, dtype='float32')
            dimension = bge_array.shape[1]
            self.visa_faiss_index_bge = faiss.IndexFlatIP(dimension)
            faiss.normalize_L2(bge_array)
            self.visa_faiss_index_bge.add(bge_array)
            print(f"✓ Built BGE FAISS index ({dimension}D, {len(bge_embeddings)} vectors)")

    def search_visa_semantic(
        self,
        query: str,
        model: str = 'sbert',
        top_k: int = 10
    ) -> List[Dict]:
        """
        Search visa requirements using FAISS semantic similarity.

        Args:
            query: User search query (e.g., "visa from Egypt to France")
            model: 'sbert' or 'bge'
            top_k: Number of results to return

        Returns:
            List of visa requirement dictionaries with similarity scores
        """
        # Check if index is built
        if model == 'sbert' and self.visa_faiss_index_sbert is None:
            raise ValueError("SBERT FAISS index not built. Call build_visa_faiss_index() first.")
        if model == 'bge' and self.visa_faiss_index_bge is None:
            raise ValueError("BGE FAISS index not built. Call build_visa_faiss_index() first.")

        # Step 1: Embed the query
        if model == 'sbert':
            query_embedding = self.sbert_model.encode(query)
            faiss_index = self.visa_faiss_index_sbert
        else:
            query_embedding = self.bge_model.encode(query)
            faiss_index = self.visa_faiss_index_bge

        # Normalize query embedding for cosine similarity
        query_embedding = query_embedding.reshape(1, -1).astype('float32')
        faiss.normalize_L2(query_embedding)

        # Step 2: Search FAISS index
        distances, indices = faiss_index.search(query_embedding, top_k)

        # Step 3: Convert to results
        results = []
        for i, (idx, score) in enumerate(zip(indices[0], distances[0])):
            if idx == -1:  # No more results
                break

            visa_data = self.visa_id_to_data[idx]
            results.append({
                'from_country': visa_data['from_country'],
                'to_country': visa_data['to_country'],
                'visa_type': visa_data['visa_type'],
                'similarity_score': float(score),
                'rank': i + 1
            })

        return results

    # ========================================================================
    # COMBINED SEARCH (Both Hotels and Visas)
    # ========================================================================

    def search_combined(
        self,
        query: str,
        search_hotels: bool = True,
        search_visas: bool = True,
        model: str = 'sbert',
        top_k: int = 10
    ) -> Dict[str, List[Dict]]:
        """
        Search both hotels and visa requirements.

        Args:
            query: User search query
            search_hotels: Whether to search hotels
            search_visas: Whether to search visas
            model: 'sbert' or 'bge'
            top_k: Number of results per category

        Returns:
            Dictionary with 'hotels' and 'visas' keys
        """
        results = {}

        if search_hotels:
            results['hotels'] = self.search_hotels_semantic(query, model, top_k)

        if search_visas:
            results['visas'] = self.search_visa_semantic(query, model, top_k)

        return results

    # ========================================================================
    # UTILITY
    # ========================================================================

    def save_visa_index(self, filepath: str = 'visa_faiss_index.pkl'):
        """Save FAISS index and metadata to disk."""
        data = {
            'visa_id_to_data': self.visa_id_to_data,
            'index_to_visa_id': self.index_to_visa_id
        }

        # Save FAISS indexes
        if self.visa_faiss_index_sbert:
            faiss.write_index(self.visa_faiss_index_sbert, 'visa_sbert.index')
        if self.visa_faiss_index_bge:
            faiss.write_index(self.visa_faiss_index_bge, 'visa_bge.index')

        # Save metadata
        with open(filepath, 'wb') as f:
            pickle.dump(data, f)

        print(f"✓ Saved FAISS index to {filepath}")

    def load_visa_index(self, filepath: str = 'visa_faiss_index.pkl'):
        """Load FAISS index and metadata from disk."""
        # Load metadata
        with open(filepath, 'rb') as f:
            data = pickle.load(f)

        self.visa_id_to_data = data['visa_id_to_data']
        self.index_to_visa_id = data['index_to_visa_id']

        # Load FAISS indexes
        if os.path.exists('visa_sbert.index'):
            self.visa_faiss_index_sbert = faiss.read_index('visa_sbert.index')
            print("✓ Loaded SBERT FAISS index")

        if os.path.exists('visa_bge.index'):
            self.visa_faiss_index_bge = faiss.read_index('visa_bge.index')
            print("✓ Loaded BGE FAISS index")

    def close(self):
        """Close Neo4j connection."""
        self.driver.close()


# ============================================================================
# Example Usage
# ============================================================================

def main():
    """Example usage of enhanced semantic search."""

    # Initialize
    searcher = EnhancedSemanticSearch(
        neo4j_uri="neo4j://127.0.0.1:7687",
        neo4j_user="neo4j",
        neo4j_password="Omarino2003"
    )

    try:
        # Build FAISS index for visa requirements (only need to do this once)
        searcher.build_visa_faiss_index(model='both')

        # Example 1: Search hotels
        print("\n" + "="*80)
        print("EXAMPLE 1: Hotel Search")
        print("="*80)

        hotel_query = "luxury hotel with spa and excellent cleanliness"
        hotel_results = searcher.search_hotels_semantic(
            query=hotel_query,
            model='sbert',
            top_k=5
        )

        print(f"\nQuery: {hotel_query}")
        print("\nTop 5 Hotels:")
        for i, hotel in enumerate(hotel_results, 1):
            print(f"{i}. {hotel['hotel_name']} - {hotel['city']}, {hotel['country']}")
            print(f"   Similarity: {hotel['similarity_score']:.4f}")
            print(f"   Rating: {hotel['avg_score']/2:.1f}/5")

        # Example 2: Search visa requirements
        print("\n" + "="*80)
        print("EXAMPLE 2: Visa Search")
        print("="*80)

        visa_query = "visa requirements from Egypt to France"
        visa_results = searcher.search_visa_semantic(
            query=visa_query,
            model='sbert',
            top_k=5
        )

        print(f"\nQuery: {visa_query}")
        print("\nTop 5 Visa Requirements:")
        for i, visa in enumerate(visa_results, 1):
            print(f"{i}. {visa['from_country']} → {visa['to_country']}")
            print(f"   Visa Type: {visa['visa_type']}")
            print(f"   Similarity: {visa['similarity_score']:.4f}")

        # Example 3: Combined search
        print("\n" + "="*80)
        print("EXAMPLE 3: Combined Search")
        print("="*80)

        combined_query = "travel to Paris with visa"
        combined_results = searcher.search_combined(
            query=combined_query,
            search_hotels=True,
            search_visas=True,
            model='sbert',
            top_k=3
        )

        print(f"\nQuery: {combined_query}")

        if 'hotels' in combined_results:
            print("\nHotels:")
            for hotel in combined_results['hotels']:
                print(f"  - {hotel['hotel_name']} ({hotel['similarity_score']:.4f})")

        if 'visas' in combined_results:
            print("\nVisa Requirements:")
            for visa in combined_results['visas']:
                print(f"  - {visa['from_country']} → {visa['to_country']} ({visa['similarity_score']:.4f})")

        # Save index for future use
        searcher.save_visa_index()

    finally:
        searcher.close()


if __name__ == "__main__":
    main()

Loading embedding models...
✓ Initialization complete

Building FAISS index for visa requirements...
Found 114 visa requirements
✓ Built SBERT FAISS index (384D, 114 vectors)
✓ Built BGE FAISS index (1024D, 114 vectors)

EXAMPLE 1: Hotel Search

Query: luxury hotel with spa and excellent cleanliness

Top 5 Hotels:
1. The Golden Oasis - Dubai, United Arab Emirates
   Similarity: 0.7696
   Rating: 4.5/5
2. The Bosphorus Inn - Istanbul, Turkey
   Similarity: 0.7414
   Rating: 4.5/5
3. Berlin Mitte Elite - Berlin, Germany
   Similarity: 0.7408
   Rating: 4.5/5
4. L'Étoile Palace - Paris, France
   Similarity: 0.7407
   Rating: 4.5/5
5. Colosseum Gardens - Rome, Italy
   Similarity: 0.7391
   Rating: 4.5/5

EXAMPLE 2: Visa Search

Query: visa requirements from Egypt to France

Top 5 Visa Requirements:
1. Egypt → France
   Visa Type: Tourist Visa
   Similarity: 0.8587
2. Egypt → Spain
   Visa Type: Tourist Visa
   Similarity: 0.6503
3. Egypt → Italy
   Visa Type: Tourist Visa
   Similarity: 

LLM Layer

OLD COMBINER: Does not use Cypher AND Semantic

In [57]:
# =============================================================================
# CELL 1: Result Combiner Class
# =============================================================================
# This cell combines results from baseline (Cypher) and embeddings (semantic search)

from typing import Dict, List, Any, Tuple

class ResultCombiner:
    """
    Combines results from baseline Cypher queries and semantic similarity search.
    Handles deduplication, ranking, and merging of information.
    """
    
    @staticmethod
    def combine_results(
        cypher_results: List[Dict],
        semantic_results: List[Dict],
        max_results: int = 10
    ) -> Tuple[List[Dict], Dict]:
        """
        Combine and deduplicate results from both retrieval methods.
        
        Args:
            cypher_results: Results from Cypher queries (baseline)
            semantic_results: Results from semantic similarity search
            max_results: Maximum number of results to return
            
        Returns:
            Tuple of (combined_results, metadata)
        """
        # Create a dictionary to track unique hotels
        hotel_map = {}
        
        # Process Cypher results first (exact matches have priority)
        for idx, result in enumerate(cypher_results):
            hotel_id = result.get('hotel_id') or result.get('hotel_name')
            if hotel_id and hotel_id not in hotel_map:
                hotel_map[hotel_id] = {
                    **result,
                    'source': 'cypher',
                    'cypher_rank': idx + 1,
                    'semantic_rank': None,
                    'similarity_score': None
                }
        
        # Add semantic results, merge if hotel already exists
        for idx, result in enumerate(semantic_results):
            hotel_id = result.get('hotel_id') or result.get('hotel_name')
            if hotel_id:
                if hotel_id in hotel_map:
                    # Hotel found in both - merge information
                    hotel_map[hotel_id]['source'] = 'both'
                    hotel_map[hotel_id]['semantic_rank'] = idx + 1
                    hotel_map[hotel_id]['similarity_score'] = result.get('similarity_score')
                else:
                    # New hotel from semantic search
                    hotel_map[hotel_id] = {
                        **result,
                        'source': 'semantic',
                        'cypher_rank': None,
                        'semantic_rank': idx + 1
                    }
        
        # Convert to list and sort by priority
        combined = list(hotel_map.values())
        
        # Sorting priority: both > cypher > semantic
        def sort_key(hotel):
            if hotel['source'] == 'both':
                return (0, hotel['cypher_rank'] or 999, -(hotel['similarity_score'] or 0))
            elif hotel['source'] == 'cypher':
                return (1, hotel['cypher_rank'], 0)
            else:  # semantic
                return (2, hotel['semantic_rank'], -(hotel['similarity_score'] or 0))
        
        combined.sort(key=sort_key)
        combined = combined[:max_results]
        
        # Create metadata
        metadata = {
            'total_cypher_results': len(cypher_results),
            'total_semantic_results': len(semantic_results),
            'combined_unique_results': len(hotel_map),
            'returned_results': len(combined),
            'results_in_both': sum(1 for h in combined if h['source'] == 'both'),
            'results_cypher_only': sum(1 for h in combined if h['source'] == 'cypher'),
            'results_semantic_only': sum(1 for h in combined if h['source'] == 'semantic')
        }
        
        return combined, metadata

NEW RESULT COMBINER: Uses Enhanced Semantic Search.

In [103]:
# =============================================================================
# CELL: Enhanced Result Combiner (Replace Your Existing ResultCombiner)
# =============================================================================

from typing import Dict, List, Any, Tuple

class ResultCombiner:
    """
    Combines results from baseline (Cypher) and embeddings (semantic search).
    Handles BOTH hotels AND visas intelligently.
    """
    
    @staticmethod
    def combine_results(
        cypher_results: List[Dict],
        semantic_results: List[Dict],
        query_type: str = None,
        intent: str = None,
        max_results: int = 10
    ) -> Tuple[List[Dict], Dict]:
        """
        Intelligently combine results based on query type.
        
        Args:
            cypher_results: Results from Cypher queries (baseline)
            semantic_results: Results from semantic similarity search
            query_type: 'hotel' or 'visa' (auto-detected if not provided)
            intent: The classified intent (optional)
            max_results: Maximum number of results to return
            
        Returns:
            Tuple of (combined_results, metadata)
        """
        # Auto-detect query type if not provided
        if not query_type:
            if intent and 'VISA' in intent:
                query_type = 'visa'
            else:
                query_type = 'hotel'
        
        # Route to appropriate combiner
        if query_type == 'visa':
            return ResultCombiner._combine_visa_results(
                cypher_results, semantic_results, max_results
            )
        else:
            return ResultCombiner._combine_hotel_results(
                cypher_results, semantic_results, max_results
            )
    
    @staticmethod
    def _combine_hotel_results(
        cypher_results: List[Dict],
        semantic_results: List[Dict],
        max_results: int
    ) -> Tuple[List[Dict], Dict]:
        """
        Combine hotel results from both retrieval methods.
        """
        hotel_map = {}
        
        # Process Cypher results first (exact matches have priority)
        for idx, result in enumerate(cypher_results):
            hotel_id = result.get('hotel_id') or result.get('hotel_name')
            if hotel_id and hotel_id not in hotel_map:
                hotel_map[hotel_id] = {
                    **result,
                    'source': 'cypher',
                    'cypher_rank': idx + 1,
                    'semantic_rank': None,
                    'similarity_score': None
                }
        
        # Add semantic results, merge if hotel already exists
        for idx, result in enumerate(semantic_results):
            hotel_id = result.get('hotel_id') or result.get('hotel_name')
            if hotel_id:
                if hotel_id in hotel_map:
                    # Hotel found in BOTH - merge information
                    hotel_map[hotel_id]['source'] = 'both'
                    hotel_map[hotel_id]['semantic_rank'] = idx + 1
                    hotel_map[hotel_id]['similarity_score'] = result.get('similarity_score')
                else:
                    # New hotel from semantic search only
                    hotel_map[hotel_id] = {
                        **result,
                        'source': 'semantic',
                        'cypher_rank': None,
                        'semantic_rank': idx + 1
                    }
        
        # Convert to list and sort by priority
        combined = list(hotel_map.values())
        
        # Sorting priority: both > cypher > semantic
        def sort_key(hotel):
            if hotel['source'] == 'both':
                return (0, hotel['cypher_rank'] or 999, -(hotel['similarity_score'] or 0))
            elif hotel['source'] == 'cypher':
                return (1, hotel['cypher_rank'], 0)
            else:  # semantic
                return (2, hotel['semantic_rank'], -(hotel['similarity_score'] or 0))
        
        combined.sort(key=sort_key)
        combined = combined[:max_results]
        
        # Create metadata
        metadata = {
            'query_type': 'hotel',
            'total_cypher_results': len(cypher_results),
            'total_semantic_results': len(semantic_results),
            'combined_unique_results': len(hotel_map),
            'returned_results': len(combined),
            'results_in_both': sum(1 for h in combined if h['source'] == 'both'),
            'results_cypher_only': sum(1 for h in combined if h['source'] == 'cypher'),
            'results_semantic_only': sum(1 for h in combined if h['source'] == 'semantic'),
            'combination_strategy': 'hotel_merge_and_rank'
        }
        
        return combined, metadata
    
    @staticmethod
    def _combine_visa_results(
        cypher_results: List[Dict],
        semantic_results: List[Dict],
        max_results: int
    ) -> Tuple[List[Dict], Dict]:
        """
        Combine visa results from both retrieval methods.
        """
        visa_map = {}
        
        # Process Cypher results first
        for idx, result in enumerate(cypher_results):
            # Create unique key for visa requirement
            from_c = result.get('from_country', '').lower().replace(' ', '_')
            to_c = result.get('to_country', '').lower().replace(' ', '_')
            visa_key = f"{from_c}_{to_c}"
            
            if visa_key and visa_key != '_' and visa_key not in visa_map:
                visa_map[visa_key] = {
                    **result,
                    'source': 'cypher',
                    'cypher_rank': idx + 1,
                    'semantic_rank': None,
                    'similarity_score': None
                }
        
        # Add semantic results, merge if visa already exists
        for idx, result in enumerate(semantic_results):
            from_c = result.get('from_country', '').lower().replace(' ', '_')
            to_c = result.get('to_country', '').lower().replace(' ', '_')
            visa_key = f"{from_c}_{to_c}"
            
            if visa_key and visa_key != '_':
                if visa_key in visa_map:
                    # Visa found in BOTH - merge information
                    visa_map[visa_key]['source'] = 'both'
                    visa_map[visa_key]['semantic_rank'] = idx + 1
                    visa_map[visa_key]['similarity_score'] = result.get('similarity_score')
                else:
                    # New visa from semantic search only
                    visa_map[visa_key] = {
                        **result,
                        'source': 'semantic',
                        'cypher_rank': None,
                        'semantic_rank': idx + 1
                    }
        
        # Convert to list and sort by priority
        combined = list(visa_map.values())
        
        # Sorting priority: both > cypher > semantic
        def sort_key(visa):
            if visa['source'] == 'both':
                return (0, visa['cypher_rank'] or 999, -(visa['similarity_score'] or 0))
            elif visa['source'] == 'cypher':
                return (1, visa['cypher_rank'], 0)
            else:  # semantic
                return (2, visa['semantic_rank'], -(visa['similarity_score'] or 0))
        
        combined.sort(key=sort_key)
        combined = combined[:max_results]
        
        # Create metadata
        metadata = {
            'query_type': 'visa',
            'total_cypher_results': len(cypher_results),
            'total_semantic_results': len(semantic_results),
            'combined_unique_results': len(visa_map),
            'returned_results': len(combined),
            'results_in_both': sum(1 for v in combined if v['source'] == 'both'),
            'results_cypher_only': sum(1 for v in combined if v['source'] == 'cypher'),
            'results_semantic_only': sum(1 for v in combined if v['source'] == 'semantic'),
            'combination_strategy': 'visa_merge_and_rank'
        }
        
        return combined, metadata

In [104]:
# =============================================================================
# CELL 2: Enhanced Prompt Builder Class
# =============================================================================

class PromptBuilder:
    """
    Builds structured prompts with context, persona, and task components.
    Handles BOTH hotel and visa queries.
    """
    
    @staticmethod
    def build_prompt(
        user_query: str,
        combined_results: List[Dict],
        metadata: Dict,
        query_type: str = None,
        persona: str = None
    ) -> str:
        """
        Build a structured prompt for the LLM.
        Auto-detects query type and routes appropriately.
        """
        # Auto-detect query type from metadata
        if query_type is None:
            query_type = metadata.get('query_type', 'hotel')
        
        # Route to appropriate builder
        if query_type == 'visa':
            return PromptBuilder._build_visa_prompt(
                user_query, combined_results, metadata, persona
            )
        else:
            return PromptBuilder._build_hotel_prompt(
                user_query, combined_results, metadata, persona
            )
    
    @staticmethod
    def _build_hotel_prompt(
        user_query: str,
        combined_results: List[Dict],
        metadata: Dict,
        persona: str = None
    ) -> str:
        """Build prompt for hotel queries."""
        if persona is None:
            persona = "helpful hotel travel assistant"
        
        context = PromptBuilder._format_hotel_context(combined_results, metadata)
        
        prompt = f"""You are a {persona} with access to a knowledge graph of hotels, reviews, and travel information.

CONTEXT FROM KNOWLEDGE GRAPH:
{context}

RETRIEVAL METADATA:
- Total results found: {metadata['combined_unique_results']}
- Results from structured queries: {metadata['total_cypher_results']}
- Results from semantic search: {metadata['total_semantic_results']}
- Results appearing in both: {metadata['results_in_both']}

USER QUESTION:
{user_query}

TASK:
Answer the user's question using ONLY the information provided in the context above. 

INSTRUCTIONS:
1. Base your answer solely on the retrieved hotel information
2. If the context doesn't contain enough information, say so clearly
3. Cite specific hotel names and details when possible
4. Be concise but informative
5. If multiple hotels match, mention the top 3-5 options
6. Include relevant ratings, locations, and key features
7. Do not make up or hallucinate information not present in the context

ANSWER:"""
        
        return prompt
    
    @staticmethod
    def _build_visa_prompt(
        user_query: str,
        combined_results: List[Dict],
        metadata: Dict,
        persona: str = None
    ) -> str:
        """Build prompt for visa queries."""
        if persona is None:
            persona = "helpful visa and travel requirements assistant"
        
        context = PromptBuilder._format_visa_context(combined_results, metadata)
        
        prompt = f"""You are a {persona} with access to a knowledge graph of visa requirements and travel regulations.

CONTEXT FROM KNOWLEDGE GRAPH:
{context}

RETRIEVAL METADATA:
- Total results found: {metadata['combined_unique_results']}
- Results from structured queries: {metadata['total_cypher_results']}
- Results from semantic search: {metadata['total_semantic_results']}
- Results appearing in both: {metadata['results_in_both']}

USER QUESTION:
{user_query}

TASK:
Answer the user's question about visa requirements using ONLY the information provided in the context above.

INSTRUCTIONS:
1. Base your answer solely on the retrieved visa information
2. Clearly state whether a visa is required or not
3. Specify the visa type if applicable
4. Be precise about the countries involved
5. If multiple country pairs are relevant, list them clearly
6. Do not make up or hallucinate visa requirements
7. If information is missing, state this clearly

ANSWER:"""
        
        return prompt
    
    @staticmethod
    def _format_hotel_context(results: List[Dict], metadata: Dict) -> str:
        """Format hotel results into readable context."""
        if not results:
            return "No relevant hotels found in the knowledge graph."
        
        context_parts = []
        
        for idx, hotel in enumerate(results, 1):
            hotel_info = []
            hotel_info.append(f"\n{idx}. {hotel.get('hotel_name', 'Unknown Hotel')}")
            
            # Location
            city = hotel.get('city', '')
            country = hotel.get('country', '')
            if city or country:
                hotel_info.append(f"   Location: {city}, {country}")
            
            # Ratings
            if 'star_rating' in hotel:
                hotel_info.append(f"   Star Rating: {hotel['star_rating']} stars")
            
            if 'avg_score' in hotel:
                hotel_info.append(f"   Average Score: {hotel['avg_score']:.1f}/10")
            elif 'overall_avg_score' in hotel:
                hotel_info.append(f"   Average Score: {hotel['overall_avg_score']:.1f}/10")
            
            # Features
            if 'cleanliness' in hotel:
                hotel_info.append(f"   Cleanliness: {hotel['cleanliness']:.1f}/10")
            if 'comfort' in hotel:
                hotel_info.append(f"   Comfort: {hotel['comfort']:.1f}/10")
            if 'facilities' in hotel:
                hotel_info.append(f"   Facilities: {hotel['facilities']:.1f}/10")
            
            # Source info
            source_info = []
            if hotel['source'] == 'both':
                source_info.append("Found in structured search AND semantic similarity")
            elif hotel['source'] == 'cypher':
                source_info.append("Found in structured search")
            else:
                source_info.append("Found in semantic similarity search")
            
            if hotel.get('similarity_score'):
                source_info.append(f"Similarity: {hotel['similarity_score']:.3f}")
            
            hotel_info.append(f"   [Source: {', '.join(source_info)}]")
            
            context_parts.append('\n'.join(hotel_info))
        
        return '\n'.join(context_parts)
    
    @staticmethod
    def _format_visa_context(results: List[Dict], metadata: Dict) -> str:
        """Format visa results into readable context."""
        if not results:
            return "No relevant visa requirements found in the knowledge graph."
        
        context_parts = []
        
        for idx, visa in enumerate(results, 1):
            visa_info = []
            from_country = visa.get('from_country', 'Unknown')
            to_country = visa.get('to_country', 'Unknown')
            visa_type = visa.get('visa_type', 'Unknown')
            
            visa_info.append(f"\n{idx}. Travel from {from_country} to {to_country}")
            visa_info.append(f"   Visa Required: YES")
            visa_info.append(f"   Visa Type: {visa_type}")
            
            # Source info
            source_info = []
            if visa['source'] == 'both':
                source_info.append("Found in structured search AND semantic similarity")
            elif visa['source'] == 'cypher':
                source_info.append("Found in structured search")
            else:
                source_info.append("Found in semantic similarity search")
            
            if visa.get('similarity_score'):
                source_info.append(f"Similarity: {visa['similarity_score']:.3f}")
            
            visa_info.append(f"   [Source: {', '.join(source_info)}]")
            
            context_parts.append('\n'.join(visa_info))
        
        return '\n'.join(context_parts)
    
    # Backward compatibility
    @staticmethod
    def build_hotel_assistant_prompt(user_query, combined_results, metadata, persona=None):
        """Backward compatible method."""
        return PromptBuilder.build_prompt(
            user_query, combined_results, metadata, 'hotel', persona
        )

In [105]:
# =============================================================================
# CELL 3: LLM Client Class
# =============================================================================
# This cell handles communication with different LLM models

import time

class LLMClient:
    """
    Handles communication with different LLM providers.
    Supports multiple models for comparison via Groq API.
    """
    
    def __init__(self, groq_api_key: str):
        """
        Initialize LLM client with Groq API.
        
        Args:
            groq_api_key: API key for Groq (provides free access to multiple models)
        """
        self.groq_client = Groq(api_key=groq_api_key)
        
        # Define available models for comparison
        self.models = {
            'llama-3.1-8b': {
                'name': 'llama-3.1-8b-instant',
                'provider': 'groq',
                'description': 'Fast, efficient, good for quick responses'
            },
            'llama-3.1-70b': {
                'name': 'llama-3.1-70b-versatile',
                'provider': 'groq',
                'description': 'Larger model, better reasoning, more accurate'
            },
            'gemma-2-9b': {
                'name': 'gemma2-9b-it',
                'provider': 'groq',
                'description': 'Google Gemma, good balance of speed and quality'
            }
        }
    
    def generate_response(
        self,
        prompt: str,
        model_key: str = 'llama-3.1-8b',
        temperature: float = 0.3,
        max_tokens: int = 1000
    ) -> Dict[str, Any]:
        """
        Generate response from LLM using the structured prompt.
        
        Args:
            prompt: The full structured prompt (context + persona + task)
            model_key: Which model to use
            temperature: Sampling temperature (0.0 = deterministic, 1.0 = creative)
            max_tokens: Maximum tokens to generate
            
        Returns:
            Dictionary containing response and metadata
        """
        if model_key not in self.models:
            raise ValueError(f"Unknown model: {model_key}. Available: {list(self.models.keys())}")
        
        model_info = self.models[model_key]
        start_time = time.time()
        
        try:
            response = self.groq_client.chat.completions.create(
                model=model_info['name'],
                messages=[
                    {
                        "role": "system",
                        "content": "You are a helpful hotel travel assistant. Answer questions based only on the provided context from the knowledge graph."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            
            elapsed_time = time.time() - start_time
            
            return {
                'success': True,
                'answer': response.choices[0].message.content,
                'model': model_key,
                'model_name': model_info['name'],
                'response_time': elapsed_time,
                'tokens_used': response.usage.total_tokens if hasattr(response, 'usage') else None,
                'finish_reason': response.choices[0].finish_reason
            }
        
        except Exception as e:
            elapsed_time = time.time() - start_time
            return {
                'success': False,
                'error': str(e),
                'model': model_key,
                'response_time': elapsed_time
            }
    
    def get_available_models(self) -> Dict[str, Dict]:
        """Return information about available models."""
        return self.models

In [106]:
# =============================================================================
# CELL 4: Complete Graph-RAG Pipeline
# =============================================================================
# This cell orchestrates all components together

class GraphRAGPipeline:
    """
    Complete Graph-RAG pipeline that orchestrates all components.
    Implements: Input → Retrieval (Baseline + Embeddings) → LLM → Output
    """
    
    def __init__(
        self,
        entity_extractor,
        intent_classifier_func,
        query_mapper,
        query_executor,
        semantic_searcher,
        llm_client: LLMClient
    ):
        """
        Initialize the complete Graph-RAG pipeline.
        
        Args:
            entity_extractor: EntityExtractor instance
            intent_classifier_func: classify_hotel_intent function
            query_mapper: QueryMapper instance
            query_executor: QueryExecutor instance
            semantic_searcher: HotelSemanticSearch instance
            llm_client: LLMClient instance
        """
        self.entity_extractor = entity_extractor
        self.classify_intent = intent_classifier_func
        self.query_mapper = query_mapper
        self.query_executor = query_executor
        self.semantic_searcher = semantic_searcher
        self.llm_client = llm_client
        # Remove this line - no need to instantiate
        # self.result_combiner = ResultCombiner()
        # self.prompt_builder = PromptBuilder()
    
    def process_query(
        self,
        user_query: str,
        model_key: str = 'llama-3.1-8b',
        use_baseline: bool = True,
        use_embeddings: bool = True,
        semantic_model: str = 'sbert',
        top_k: int = 10,
        temperature: float = 0.3,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        Process a user query through the complete Graph-RAG pipeline.
        """
        pipeline_start = time.time()
        
        if verbose:
            print(f"Processing query: '{user_query}'")
            print("="*80)
        
        # Step 1: Intent Classification
        if verbose:
            print("Step 1: Classifying intent...")
        intent = self.classify_intent(user_query)
        if verbose:
            print(f"  → Intent: {intent}\n")
        
        # Step 2: Entity Extraction
        if verbose:
            print("Step 2: Extracting entities...")
        entities = self.entity_extractor.extract_entities(user_query)
        if verbose:
            print(f"  → Entities: {entities}\n")
        
        # Step 3a: Baseline Retrieval (Cypher)
        cypher_results = []
        if use_baseline:
            if verbose:
                print("Step 3a: Retrieving with Cypher queries (Baseline)...")
            try:
                params = self.query_mapper.map_entities_to_parameters(intent, entities)
                cypher_results = self.query_executor.execute_query(intent, params)
                if verbose:
                    print(f"  → Found {len(cypher_results)} results from Cypher\n")
            except Exception as e:
                if verbose:
                    print(f"  → Cypher query failed: {e}\n")
        
        # Step 3b: Embedding-based Retrieval
        semantic_results = []
        if use_embeddings:
            if verbose:
                print("Step 3b: Retrieving with semantic search (Embeddings)...")
            try:
                # Detect if this is a visa query based on intent
                if 'VISA' in intent:
                    if verbose:
                        print("  → Using visa semantic search...")
                    semantic_results = self.semantic_searcher.search_visa_semantic(
                        query=user_query,
                        model=semantic_model,
                        top_k=top_k
                    )
                else:
                    if verbose:
                        print("  → Using hotel semantic search...")
                    
                    # EXTRACT FILTERS FROM ENTITIES
                    city_filter = entities.get('city', [None])[0] if entities.get('city') else None
                    country_filter = entities.get('country', [None])[0] if entities.get('country') else None
                    
                    if verbose and (city_filter or country_filter):
                        print(f"  → Applying filters: city={city_filter}, country={country_filter}")
                    
                    semantic_results = self.semantic_searcher.search_hotels_semantic(
                        query=user_query,
                        model=semantic_model,
                        top_k=top_k,
                        city_filter=city_filter,  # ADD THIS
                        country_filter=country_filter  # ADD THIS
                    )
                if verbose:
                    print(f"  → Found {len(semantic_results)} results from semantic search\n")
            except Exception as e:
                if verbose:
                    print(f"  → Semantic search failed: {e}\n")
        
        # Step 4: Combine Results - CALL AS STATIC METHOD
        if verbose:
            print("Step 4: Combining results from both methods...")
        combined_results, metadata = ResultCombiner.combine_results(
            cypher_results=cypher_results,
            semantic_results=semantic_results,
            intent=intent,
            max_results=top_k
        )
        if verbose:
            print(f"  → Strategy: {metadata['combination_strategy']}")
            print(f"  → Combined {metadata['combined_unique_results']} unique results")
            print(f"  → {metadata['results_in_both']} found in BOTH methods")
            print(f"  → {metadata['results_cypher_only']} only in Cypher")
            print(f"  → {metadata['results_semantic_only']} only in semantic search")
            if 'note' in metadata:
                print(f"  → Note: {metadata['note']}")
            print()
        
        # Step 5: Build Structured Prompt - CALL AS STATIC METHOD
        if verbose:
            print("Step 5: Building structured prompt (Context + Persona + Task)...")

        prompt = PromptBuilder.build_prompt(
            user_query=user_query,
            combined_results=combined_results,
            metadata=metadata
        )

        if verbose:
            print(f"  → Prompt created ({len(prompt)} characters)")
            print(f"  → Query type: {metadata.get('query_type', 'hotel')}\n")
        
        # Step 6: Generate Answer with LLM
        if verbose:
            print(f"Step 6: Generating answer with {model_key}...")
        llm_response = self.llm_client.generate_response(
            prompt=prompt,
            model_key=model_key,
            temperature=temperature
        )
        
        pipeline_time = time.time() - pipeline_start
        
        if verbose:
            print(f"  → Answer generated in {llm_response['response_time']:.2f}s")
            print(f"\nTotal pipeline time: {pipeline_time:.2f}s")
            print("="*80)
        
        return {
            'query': user_query,
            'intent': intent,
            'entities': entities,
            'cypher_results': cypher_results,
            'semantic_results': semantic_results,
            'combined_results': combined_results,
            'retrieval_metadata': metadata,
            'prompt': prompt,
            'llm_response': llm_response,
            'total_pipeline_time': pipeline_time,
            'retrieval_config': {
                'baseline_used': use_baseline,
                'embeddings_used': use_embeddings,
                'semantic_model': semantic_model if use_embeddings else None,
                'llm_model': model_key
            }
        }
    
    def compare_llm_models(
        self,
        user_query: str,
        model_keys: List[str] = None
    ) -> Dict[str, Any]:
        """
        Compare responses from different LLM models (Requirement 3c).
        
        Args:
            user_query: User's question
            model_keys: List of models to compare
            
        Returns:
            Comparison results from each model
        """
        if model_keys is None:
            model_keys = ['llama-3.1-8b', 'llama-3.1-70b', 'gemma-2-9b']
        
        print(f"Comparing {len(model_keys)} LLM models...")
        print("="*80)
        
        # Retrieve information once (same context for all models)
        intent = self.classify_intent(user_query)
        entities = self.entity_extractor.extract_entities(user_query)
        
        params = self.query_mapper.map_entities_to_parameters(intent, entities)
        cypher_results = self.query_executor.execute_query(intent, params)
        
        # Detect query type for semantic search
        if 'VISA' in intent:
            semantic_results = self.semantic_searcher.search_visa_semantic(
                query=user_query, model='sbert', top_k=10
            )
        else:
            semantic_results = self.semantic_searcher.search_hotels_semantic(
                query=user_query, model='sbert', top_k=10
            )
        
        combined_results, metadata = ResultCombiner.combine_results(
            cypher_results, semantic_results, intent=intent
        )
        
        prompt = PromptBuilder.build_prompt(
            user_query, combined_results, metadata
        )
        
        # Generate responses from each model
        results = {}
        for model_key in model_keys:
            print(f"\nGenerating answer with {model_key}...")
            llm_response = self.llm_client.generate_response(
                prompt=prompt,
                model_key=model_key,
                temperature=0.3
            )
            
            results[model_key] = {
                'query': user_query,
                'llm_response': llm_response,
                'retrieval_metadata': metadata,
                'combined_results_count': len(combined_results)
            }
            
            if llm_response['success']:
                print(f"  ✓ Response time: {llm_response['response_time']:.2f}s")
                print(f"  ✓ Tokens: {llm_response.get('tokens_used', 'N/A')}")
        
        print("\n" + "="*80)
        return results

In [107]:
# =============================================================================
# CELL 5: Initialize Complete Pipeline
# =============================================================================
# This cell demonstrates how to use the complete pipeline

# Re-initialize database connections (in case they were closed)
executor = QueryExecutor(
    uri="neo4j://127.0.0.1:7687",
    user="neo4j",
    password="Omarino2003"
)

mapper = QueryMapper()

# Initialize LLM Client
llm_client = LLMClient(groq_api_key="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")

# Initialize ENHANCED semantic searcher (handles both hotels AND visas)
print("Initializing Enhanced Semantic Searcher...")
semantic_searcher = EnhancedSemanticSearch(
    neo4j_uri="neo4j://127.0.0.1:7687",
    neo4j_user="neo4j",
    neo4j_password="Omarino2003"
)

# Build FAISS index for visa requirements (required for visa search)
print("Building FAISS index for visa requirements...")
semantic_searcher.build_visa_faiss_index(model='both')
print("✓ FAISS index built successfully!\n")

# Create complete Graph-RAG pipeline
pipeline = GraphRAGPipeline(
    entity_extractor=entity_extractor,
    intent_classifier_func=classify_hotel_intent,
    query_mapper=mapper,
    query_executor=executor,
    semantic_searcher=semantic_searcher,  # Now using EnhancedSemanticSearch
    llm_client=llm_client
)

print("✓ Database connections refreshed!")
print("✓ Graph-RAG Pipeline initialized successfully!")
print(f"✓ Available LLM models: {list(llm_client.models.keys())}")
print("✓ Supports: Hotel search + Visa requirements")

Initializing Enhanced Semantic Searcher...
Loading embedding models...
✓ Initialization complete
Building FAISS index for visa requirements...

Building FAISS index for visa requirements...
Found 114 visa requirements
✓ Built SBERT FAISS index (384D, 114 vectors)
✓ Built BGE FAISS index (1024D, 114 vectors)
✓ FAISS index built successfully!

✓ Database connections refreshed!
✓ Graph-RAG Pipeline initialized successfully!
✓ Available LLM models: ['llama-3.1-8b', 'llama-3.1-70b', 'gemma-2-9b']
✓ Supports: Hotel search + Visa requirements


In [111]:
# =============================================================================
# CELL 6: Test Queries (Hotels and Visas)
# =============================================================================

print("="*80)
print("TESTING HOTEL QUERY")
print("="*80)

hotel_query = "Find 5-star hotels in Paris with excellent cleanliness"
hotel_result = pipeline.process_query(
    user_query=hotel_query,
    model_key='llama-3.1-8b',
    use_baseline=True,
    use_embeddings=True,
    semantic_model='sbert',
    temperature=0.3,
    verbose=True
)

print("\n" + "="*80)
print("FINAL ANSWER (HOTEL QUERY):")
print("="*80)
if hotel_result['llm_response']['success']:
    print(hotel_result['llm_response']['answer'])
else:
    print(f"Error: {hotel_result['llm_response'].get('error', 'Unknown error')}")

# Test visa query
print("\n\n" + "="*80)
print("TESTING VISA QUERY")
print("="*80)

visa_query = "Do I need a visa to travel from Egypt to France?"
visa_result = pipeline.process_query(
    user_query=visa_query,
    model_key='llama-3.1-8b',
    use_baseline=True,
    use_embeddings=True,
    semantic_model='sbert',
    temperature=0.3,
    verbose=True
)

print("\n" + "="*80)
print("FINAL ANSWER (VISA QUERY):")
print("="*80)
if visa_result['llm_response']['success']:
    print(visa_result['llm_response']['answer'])
else:
    print(f"Error: {visa_result['llm_response'].get('error', 'Unknown error')}")

# Optional: Print debug info to verify correct routing
print("\n" + "="*80)
print("DEBUG INFO:")
print("="*80)
print(f"Hotel query intent: {hotel_result['intent']}")
print(f"Hotel query type: {hotel_result['retrieval_metadata']['query_type']}")
print(f"Hotel semantic results count: {len(hotel_result['semantic_results'])}")
print(f"Hotel cypher results count: {len(hotel_result['cypher_results'])}")

print(f"\nVisa query intent: {visa_result['intent']}")
print(f"Visa query type: {visa_result['retrieval_metadata']['query_type']}")
print(f"Visa semantic results count: {len(visa_result['semantic_results'])}")
print(f"Visa cypher results count: {len(visa_result['cypher_results'])}")

TESTING HOTEL QUERY
Processing query: 'Find 5-star hotels in Paris with excellent cleanliness'
Step 1: Classifying intent...
  → Intent: HOTEL_SEARCH

Step 2: Extracting entities...
  → Entities: {'city': ['paris'], 'star_rating': ['5'], 'feature': ['cleanliness']}

Step 3a: Retrieving with Cypher queries (Baseline)...
  → Found 1 results from Cypher

Step 3b: Retrieving with semantic search (Embeddings)...
  → Using hotel semantic search...
  → Applying filters: city=paris, country=None
  → Found 1 results from semantic search

Step 4: Combining results from both methods...
  → Strategy: hotel_merge_and_rank
  → Combined 1 unique results
  → 1 found in BOTH methods
  → 0 only in Cypher
  → 0 only in semantic search

Step 5: Building structured prompt (Context + Persona + Task)...
  → Prompt created (1131 characters)
  → Query type: hotel

Step 6: Generating answer with llama-3.1-8b...
  → Answer generated in 0.63s

Total pipeline time: 0.95s

FINAL ANSWER (HOTEL QUERY):
Based on the p

In [112]:
# =============================================================================
# CELL 7: Compare Multiple LLM Models (Requirement 3c)
# =============================================================================

# Compare 3 different LLM models on the same query
comparison = pipeline.compare_llm_models(
    user_query="What are the best hotels for business travelers in Dubai?",
    model_keys=['llama-3.1-8b', 'llama-3.1-70b', 'gemma-2-9b']
)

# Display comparison results
print("\n" + "="*80)
print("LLM MODEL COMPARISON")
print("="*80)

for model, result in comparison.items():
    llm_resp = result['llm_response']
    print(f"\n{model.upper()}:")
    print(f"  Response Time: {llm_resp['response_time']:.2f}s")
    print(f"  Tokens Used: {llm_resp.get('tokens_used', 'N/A')}")
    print(f"  Success: {llm_resp['success']}")
    if llm_resp['success']:
        print(f"\n  Answer:")
        print(f"  {llm_resp['answer'][:300]}...")
    print("-"*80)


Comparing 3 LLM models...

Generating answer with llama-3.1-8b...
  ✓ Response time: 0.57s
  ✓ Tokens: 1187

Generating answer with llama-3.1-70b...

Generating answer with gemma-2-9b...


LLM MODEL COMPARISON

LLAMA-3.1-8B:
  Response Time: 0.57s
  Tokens Used: 1187
  Success: True

  Answer:
  Based on the provided hotel information, I recommend the following top 3 hotels for business travelers in Dubai:

1. **The Golden Oasis**: This 5-star hotel in Dubai boasts a 5.0-star rating and exceptional cleanliness (9.3/10), comfort (9.5/10), and facilities (9.6/10). Its high ratings suggest a w...
--------------------------------------------------------------------------------

LLAMA-3.1-70B:
  Response Time: 0.09s
  Tokens Used: N/A
  Success: False
--------------------------------------------------------------------------------

GEMMA-2-9B:
  Response Time: 0.12s
  Tokens Used: N/A
  Success: False
--------------------------------------------------------------------------------


In [113]:
# =============================================================================
# CELL 8: Test Different Retrieval Strategies
# =============================================================================

# Test with different combinations of baseline and embeddings
test_query = "Recommend a hotel for a 28-year-old female solo traveler"

print("\n" + "="*80)
print("TESTING RETRIEVAL STRATEGIES")
print("="*80)

# Strategy 1: Baseline only (Experiment 1)
print("\n1. BASELINE ONLY (Cypher Queries)")
print("-"*80)
result_baseline = pipeline.process_query(
    user_query=test_query,
    use_baseline=True,
    use_embeddings=False,
    verbose=False
)
print(f"Results found: {len(result_baseline['combined_results'])}")
print(f"Answer: {result_baseline['llm_response']['answer'][:200]}...")

# Strategy 2: Embeddings only (Experiment 2)
print("\n2. EMBEDDINGS ONLY (Semantic Search)")
print("-"*80)
result_embeddings = pipeline.process_query(
    user_query=test_query,
    use_baseline=False,
    use_embeddings=True,
    verbose=False
)
print(f"Results found: {len(result_embeddings['combined_results'])}")
print(f"Answer: {result_embeddings['llm_response']['answer'][:200]}...")

# Strategy 3: Hybrid (Both)
print("\n3. HYBRID (Baseline + Embeddings)")
print("-"*80)
result_hybrid = pipeline.process_query(
    user_query=test_query,
    use_baseline=True,
    use_embeddings=True,
    verbose=False
)
print(f"Results found: {len(result_hybrid['combined_results'])}")
print(f"Cypher only: {result_hybrid['retrieval_metadata']['results_cypher_only']}")
print(f"Semantic only: {result_hybrid['retrieval_metadata']['results_semantic_only']}")
print(f"Both: {result_hybrid['retrieval_metadata']['results_in_both']}")
print(f"Answer: {result_hybrid['llm_response']['answer'][:200]}...")

print("\n" + "="*80)


TESTING RETRIEVAL STRATEGIES

1. BASELINE ONLY (Cypher Queries)
--------------------------------------------------------------------------------
Results found: 10
Answer: Based on the provided hotel information, I would recommend the following top 3 hotels for a 28-year-old female solo traveler:

1. **The Golden Oasis** in Dubai, United Arab Emirates - This 5-star hote...

2. EMBEDDINGS ONLY (Semantic Search)
--------------------------------------------------------------------------------
Results found: 10
Answer: Based on the provided hotel information, I would recommend the following top 3 hotels for a 28-year-old female solo traveler:

1. **The Golden Oasis** in Dubai, United Arab Emirates - This 5-star hote...

3. HYBRID (Baseline + Embeddings)
--------------------------------------------------------------------------------
Results found: 10
Cypher only: 5
Semantic only: 0
Both: 5
Answer: Based on the provided hotel information, I recommend the following top 3 hotels for a 28-year